In [ ]:
# =========================
# START NEW PATIENT — HARD RESET
# Run before loading the next patient file to remove patient-specific
# variables from the global workspace and prevent carryover between runs.
# =========================

def reset_patient_state():
    to_del = [
        # Core analysis objects
        "master", "episode_tables", "review_tables", "final_sets",

        # Export tables and export bundle
        "qc_tbl", "event_tbl", "run_meta_tbl", "export_pack",

        # Baseline-related objects
        "baseline_candidates", "baseline_model_tbl", "baseline_sensitivity_tbl",
        "baseline_final_summary_tbl", "baseline_final_summary_tbl_out",
        "baseline_protocol_SBP_supine", "baseline_protocol_SBP_seated",
        "baseline_stable_clean_resting", "baseline_low_motion_clean_resting", "baseline_global_resting",

        # ABPM and diurnal summary tables
        "abpm_diagnostic_summary_tbl", "diurnal_tbl",

        # Posture feasibility outputs
        "pos_meta", "posture_feasibility_tbl", "posture_feasibility_tbl_out",

        # Protocol extraction tables
        "protocol_tbl", "protocol_summary",
        "protocol_supine_tbl", "protocol_seated_tbl", "protocol_drift_tbl",
    ]

    removed = []
    for v in to_del:
        if v in globals():
            del globals()[v]
            removed.append(v)

    print(f"Per-patient state cleared. Removed {len(removed)} variables.")

reset_patient_state()

In [ ]:
# =========================
# SECTION 0 — CONFIG + ANALYSIS CONTRACT
# =========================
import os, re, math
import numpy as np
import pandas as pd
from typing import Optional, Dict, List, Tuple

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 180)

# Version identifier for the locked analysis configuration used in this notebook.
CFG_VERSION = "LOCKED_motionFirst_anyMotion_ADOnsetGates_BPState60s_baselineSweepReady"

CFG = {
    "FREQ": "1s",

    # -------------------------
    # Core event thresholds
    # -------------------------
    "AD_RISE": 20,
    "AD_RISE_HIGH": 30,
    "OH_DROP": 20,
    "OH_HR_RISE": 5,

    # -------------------------
    # Blood pressure range thresholds
    # Descriptive only; not diagnostic criteria
    # -------------------------
    "HYPOTENSION_SBP": 90,
    "HYPOTENSION_DBP": 60,
    "HTN_SBP": 140,
    "HTN_DBP": 90,

    # Minimum duration for descriptive BP state outputs
    "BP_STATE_MIN_SEC": 60,

    # -------------------------
    # Feature extraction windows
    # -------------------------
    "BASELINE_LOCAL_SEC": 120,
    "BASELINE_LOCAL_MINP": 60,
    "HR_DELTA_SEC": 10,
    "PTT_SD_SEC": 30,
    "PTT_SD_MINP": 15,

    # -------------------------
    # Quality control thresholds
    # -------------------------
    "RR_MIN_MS": 250,
    "RR_MAX_MS": 2000,
    "RR_JUMP_MS": 200,
    "HR_RR_TOL_PCT": 20,
    "SBP_SPIKE_HARD": 30,
    "SBP_SPIKE_MOD": 15,

    # -------------------------
    # Activity thresholds
    # -------------------------
    "ABS_LOW_MOTION_MAX": 1,
    "HIGH_MOTION_FLOOR": 3,
    "VHIGH_MOTION_FLOOR": 10,

    # -------------------------
    # PTT-noise quantile thresholds
    # -------------------------
    "PTT_SD_Q": 0.95,
    "PTT_EVT_Q": 0.95,

    # -------------------------
    # Onset buffer for classification gates
    # -------------------------
    "ONSET_BUF_SEC": 10,

    # AD onset physiology gates
    "AD_ONSET_WINDOW_SEC": 20,
    "AD_ONSET_MAP_RISE": 5,
    "AD_ONSET_DBP_RISE": 3,
    "AD_MIN_CONCORDANCE_FRAC": 0.60,
    "AD_MIN_PTT_STABLE_FRAC": 0.80,

    # -------------------------
    # Context streams
    # Used for contextual interpretation only; not diagnostic gates
    # -------------------------
    "CONTEXT_FFILL_LIMIT_SEC": 6 * 3600,
    "SLEEP_LABELS": ["Sleep"],
    "WAKE_LABELS": ["Wake"],
    "SLEEP_MIN_CONTEXT_SEC": 30 * 60,

    # -------------------------
    # Episode construction
    # -------------------------
    "BRIDGE_GAP_SEC": 2,
    "PRIMARY_MIN_SEC": 5,
    "AD_SUSTAINED_MIN_SEC": 30,
    "OTHER_SUSTAINED_MIN_SEC": 10,

    # -------------------------
    # Adjudication and scoring
    # -------------------------
    "EXTRA_Intermediate-Quality_PTT_ONSET_PENALTY": 1,

    # -------------------------
    # Posture anchoring
    # -------------------------
    "UPRIGHT_LABELS": {"Standing", "Upright", "Sitting", "Seated"},
    "SUPINE_LABELS": {"Supine", "Lying", "Lie", "Recumbent"},
    "POS_TAU_SEC": 30 * 60,
    "ANCHOR_THR": 0.60,
    "OH_WINDOW_SEC": 180,
    "ANCHOR_PRE_SEC": 30,
    "ANCHOR_POST_SEC": 30,
    "POS_CONF_MIN_FOR_SUPINE_LABEL": 0.60,

    # -------------------------
    # Spike guard
    # Intended to prevent short transient spikes from triggering thresholds
    # -------------------------
    "SPIKE_GLITCH_RETURN_TOL": 5,     # mmHg
    "SPIKE_GLITCH_WINDOW_SEC": 5,     # seconds

    # -------------------------
    # Baseline method sweep
    # Defines the baseline methods summarised in sensitivity outputs
    # -------------------------
    "BASELINE_METHODS": [
        "protocol_supine",
        "stable_clean_resting",
        "low_motion_clean_resting",
        "global_resting_analyzable",
    ],

    # Default baseline method used in the single-run pipeline
    "BASELINE_ACTIVE_METHOD": "protocol_supine",
}

patient_id = "P-Demo"  # Replace with patient identifier

# =========================
# Protocol timestamps (to be defined per patient/session)
# These timestamps should correspond to the clock-times at which
# protocol BP readings were taken during the recording.
# =========================

# Supine protocol readings
supine_times = [
"12:18:02","12:19:19","12:20:33","12:21:39","12:22:51",
"12:23:51","12:24:51","12:25:51","12:27:08","12:28:19"
]
seated_times = [
"12:30:19","12:31:50","12:33:20","12:34:10","12:34:59",
"12:36:00","12:37:25","12:38:24","12:39:14","12:40:09"
]
drift_times = ["07:19:45","07:21:30","07:23:31","07:26:31","07:27:31"]

print("SECTION 0 loaded.")
print("CFG_VERSION:", CFG_VERSION)

# Analysis contract
print("Contract: QC ≠ diagnosis; context streams are descriptive only.")
print("Contract: AD_resting requires resting eligibility + onset physiology gates; failures become context_excluded.")
print("Contract: Effort-associated rises are reported as EFFORT_PRESSOR (not AD).")
print(f"Contract: OH_posture_anchored only when posture anchor strength >= {CFG['ANCHOR_THR']:.2f};")
print("          otherwise drops are reported as BP_drop_local_nonanchored (not OH diagnosis).")
print("Contract: HighBP/LowBP outputs are RANGE (descriptive) and STATE (>=60s, valid-only) — not diagnostic hypertension.")
print("Contract: Baseline sensitivity: summaries will be generated per baseline method listed in CFG['BASELINE_METHODS'].")
print("Contract: Spike-guard prevents 1–5s needle spikes from threshold triggering (verify it triggers on your device data).")

In [ ]:
# =========================
# SECTION 1 — LOAD WORKBOOK AND PARSE SHEETS
# Robust parsing for numeric, categorical, and event-based DOMINO/SOMNO exports.
# =========================

import warnings

def parse_domino_time(series: pd.Series) -> pd.Series:
    """
    Parse DOMINO/SOMNO timestamp strings using a staged approach.

    Strategy:
    1. Try known timestamp formats first for speed and stable parsing.
    2. Fall back to mixed-format parsing for heterogeneous exports.
    3. Apply a final non-dayfirst fallback if needed.

    Returns
    -------
    pd.Series
        Datetime values floored to whole seconds.
    """
    s = series.copy()

    # Preserve existing datetime-like input and standardise to second resolution.
    if np.issubdtype(s.dtype, np.datetime64):
        return pd.to_datetime(s, errors="coerce").dt.floor("s")

    s = s.astype("object")
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    # Known timestamp formats used in common exports.
    formats = [
        "%Y-%m-%d %H:%M:%S",
        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%y %H:%M:%S",
        "%Y/%m/%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
        "%d/%m/%Y %H:%M",
    ]

    remaining = out.isna()
    for fmt in formats:
        if not remaining.any():
            break
        parsed = pd.to_datetime(s[remaining], format=fmt, errors="coerce")
        out.loc[remaining] = parsed
        remaining = out.isna()

    # Fallback for heterogeneous strings not captured by known formats.
    if remaining.any():
        try:
            parsed = pd.to_datetime(s[remaining], errors="coerce", format="mixed", dayfirst=True)
        except TypeError:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                parsed = pd.to_datetime(s[remaining], errors="coerce", dayfirst=True)
        out.loc[remaining] = parsed
        remaining = out.isna()

    # Final fallback without day-first assumption.
    if remaining.any():
        try:
            parsed = pd.to_datetime(s[remaining], errors="coerce", format="mixed")
        except TypeError:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                parsed = pd.to_datetime(s[remaining], errors="coerce")
        out.loc[remaining] = parsed

    return out.dt.floor("s")


# Mapping of analysis variable names to expected workbook sheet names.
SHEET_MAP_NUMERIC = {
    "SBP": "Systolic",
    "DBP": "Diastolic",
    "MAP": "MAP",
    "HR": "Heart Rate Curve",
    "RR": "RR-Interval",
    "PTTraw": "PTT Raw",
    "Activity": "Activity",
    "SpO2": "SpO2",
}

SHEET_MAP_CATEGORICAL = {
    "Position": "Position",
    "SleepWake": "Sleep-Wake",
}

# Candidate event-sheet names to accommodate export naming variation.
EVENT_SHEET_CANDIDATES = {
    "PTT": ["PTT Events", "PTT Event", "PTT Raw Events", "PTT"],
    "SBP": ["Systolic Events", "SBP Events", "BP Events", "Systolic"],
}


def choose_input_file() -> str:
    """
    Select an input workbook.

    In Colab, prompts the user to upload an .xlsx file.
    Outside Colab, requests a local file path.
    """
    if IN_COLAB:
        print("Upload the SOMNO/DOMINO Excel workbook (.xlsx):")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        filename = next(iter(uploaded))
        path = os.path.join("/content", filename)
        print(f"Using uploaded file: {path}")
        return path
    else:
        path = input("Enter workbook path: ").strip()
        if not os.path.exists(path):
            raise FileNotFoundError(f"File not found: {path}")
        return path


def find_header_row(raw_df: pd.DataFrame, must_contain: List[str], max_rows: int = 80) -> int:
    """
    Identify the header row by searching for required keywords
    within the first part of a sheet.
    """
    must = [m.lower() for m in must_contain]
    for i in range(min(max_rows, len(raw_df))):
        vals = raw_df.iloc[i].astype(str).str.strip().str.lower().tolist()
        if all(any(m in v for v in vals) for m in must):
            return i
    raise ValueError(f"Could not find header row containing: {must_contain}")


def load_timeseries_numeric(xl: pd.ExcelFile, sheet_name: str) -> pd.DataFrame:
    """
    Load a numeric timeseries sheet with columns [Time, Value].

    Duplicate timestamps are collapsed by median value, and the output
    is indexed by time at one-second resolution.
    """
    raw = xl.parse(sheet_name=sheet_name, header=None)
    h = find_header_row(raw, must_contain=["time"])
    df = raw.iloc[h + 1:, :2].copy()
    df.columns = ["Time", "Value"]
    df["Time"] = parse_domino_time(df["Time"])
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    df = df.dropna(subset=["Time"]).copy()
    df = df.groupby("Time", as_index=False)["Value"].median().set_index("Time").sort_index()
    return df


def load_timeseries_categorical(xl: pd.ExcelFile, sheet_name: str) -> pd.DataFrame:
    """
    Load a categorical timeseries sheet with columns [Time, Value].

    Duplicate timestamps are collapsed by taking the last observed value.
    Placeholder entries are converted to missing values.
    """
    raw = xl.parse(sheet_name=sheet_name, header=None)
    h = find_header_row(raw, must_contain=["time"])
    df = raw.iloc[h + 1:, :2].copy()
    df.columns = ["Time", "Value"]
    df["Time"] = parse_domino_time(df["Time"])
    df = df.dropna(subset=["Time"]).copy()
    df["Value"] = df["Value"].astype(str).str.strip()
    df.loc[df["Value"].isin(["A", "nan", "NaN", "None", ""]), "Value"] = np.nan
    df = df.groupby("Time", as_index=True)["Value"].last().to_frame().sort_index()
    return df


def detect_event_columns(raw_df: pd.DataFrame, header_row: int) -> Tuple[int, int]:
    """
    Detect start-time and end-time columns in an event sheet header.
    """
    headers = raw_df.iloc[header_row].astype(str).str.strip().str.lower().tolist()
    start_idx = end_idx = None

    for i, h in enumerate(headers):
        if start_idx is None and ("start" in h and "time" in h):
            start_idx = i
        if end_idx is None and ("end" in h and "time" in h):
            end_idx = i

    if start_idx is None:
        for i, h in enumerate(headers):
            if "start" in h:
                start_idx = i
                break

    if end_idx is None:
        for i, h in enumerate(headers):
            if "end" in h:
                end_idx = i
                break

    if start_idx is None or end_idx is None:
        raise ValueError("Could not identify event start/end columns.")

    return start_idx, end_idx


def load_event_sheet_from_name(xl: pd.ExcelFile, sheet_name: str) -> pd.DataFrame:
    """
    Load an event sheet and return a table with [Start, End] timestamps.
    Invalid or incomplete rows are excluded.
    """
    raw = xl.parse(sheet_name=sheet_name, header=None)

    header_row = None
    for i in range(min(80, len(raw))):
        vals = raw.iloc[i].astype(str).str.strip().str.lower().tolist()
        if any("start" in v for v in vals) and any("end" in v for v in vals):
            header_row = i
            break

    if header_row is None:
        raise ValueError("Could not find event header row.")

    s_col, e_col = detect_event_columns(raw, header_row)
    df = raw.iloc[header_row + 1:, [s_col, e_col]].copy()
    df.columns = ["Start", "End"]
    df["Start"] = parse_domino_time(df["Start"])
    df["End"] = parse_domino_time(df["End"])
    df = df.dropna(subset=["Start", "End"]).copy()
    df = df[df["End"] >= df["Start"]].sort_values(["Start", "End"]).reset_index(drop=True)
    return df


def try_load_event_sheet(xl: pd.ExcelFile, candidates: List[str]) -> Optional[pd.DataFrame]:
    """
    Attempt to load an event sheet from a list of candidate names.

    If a matching sheet is found but fails parsing, the function continues
    to test remaining candidates before returning None.
    """
    last_err = None
    for name in candidates:
        if name in xl.sheet_names:
            try:
                out = load_event_sheet_from_name(xl, name)
                print(f"Loaded event sheet: {name} ({len(out)} rows)")
                return out
            except Exception as e:
                last_err = e
                print(f"Found event sheet '{name}' but could not parse it: {e}")
                continue
    return None


# ---------- Load workbook ----------
FILE_PATH = choose_input_file()
xl = pd.ExcelFile(FILE_PATH, engine="openpyxl")

# ---------- Load numeric channels ----------
ts_num: Dict[str, pd.DataFrame] = {}
for key, sheet in SHEET_MAP_NUMERIC.items():
    if sheet not in xl.sheet_names:
        raise RuntimeError(f"Missing required numeric channel '{key}' from sheet '{sheet}'")
    ts_num[key] = load_timeseries_numeric(xl, sheet)
    print(f"Loaded numeric: {key} <- '{sheet}' ({len(ts_num[key])} rows)")

# ---------- Load categorical channels ----------
ts_cat: Dict[str, pd.DataFrame] = {}
for key, sheet in SHEET_MAP_CATEGORICAL.items():
    if sheet in xl.sheet_names:
        ts_cat[key] = load_timeseries_categorical(xl, sheet)
        print(f"Loaded categorical: {key} <- '{sheet}' ({len(ts_cat[key])} rows)")
    else:
        print(f"Optional categorical missing: {key} <- '{sheet}'")

# ---------- Load event tables ----------
event_tables: Dict[str, Optional[pd.DataFrame]] = {}
for key, candidates in EVENT_SHEET_CANDIDATES.items():
    event_tables[key] = try_load_event_sheet(xl, candidates)

print("SECTION 1 loaded.")

In [ ]:
# =========================
# SECTION 2 — BUILD 1 Hz MASTER AND MAP EVENT TABLES
# =========================

def build_master_dataframe(
    ts_num: Dict[str, pd.DataFrame],
    ts_cat: Dict[str, pd.DataFrame],
    freq: str = "1s"
) -> pd.DataFrame:
    """
    Construct the master dataframe on a regular time grid.

    Numeric channels are aligned directly to the master index.
    Categorical channels are added when available; otherwise they are
    initialised as missing.
    """
    tmin = min(df.index.min() for df in ts_num.values())
    tmax = max(df.index.max() for df in ts_num.values())
    idx = pd.date_range(tmin.floor("s"), tmax.ceil("s"), freq=freq)

    master = pd.DataFrame(index=idx)

    # Add numeric channels to the master index.
    for key, df in ts_num.items():
        master[key] = df["Value"].reindex(idx)

    # Add categorical context channels when present.
    for key in ["Position", "SleepWake"]:
        if key in ts_cat:
            master[key] = ts_cat[key]["Value"].reindex(idx)
        else:
            master[key] = np.nan

    return master


def map_event_table(master: pd.DataFrame, events_df: Optional[pd.DataFrame], prefix: str) -> pd.DataFrame:
    """
    Map an event table onto the master dataframe.

    Outputs
    -------
    {prefix}_event_start : integer
        Number of event onsets occurring within each second.
    {prefix}_event_active : boolean
        True during the event interval.
    {prefix}_events_per_min : float
        Rolling 60-second sum of event starts.
    """
    start_col = f"{prefix}_event_start"
    active_col = f"{prefix}_event_active"
    density_col = f"{prefix}_events_per_min"

    master[start_col] = np.zeros(len(master), dtype=np.int32)
    master[active_col] = False

    if events_df is None or len(events_df) == 0:
        master[density_col] = pd.Series(master[start_col].values, index=master.index).rolling(60, min_periods=1).sum()
        return master

    idx = master.index

    # Standardise event boundaries to second resolution.
    starts = pd.to_datetime(events_df["Start"], errors="coerce").dt.floor("s")
    ends = pd.to_datetime(events_df["End"], errors="coerce").dt.floor("s")

    good = starts.notna() & ends.notna() & (ends >= starts)
    starts = starts[good].values
    ends = ends[good].values

    if len(starts) == 0:
        master[density_col] = pd.Series(master[start_col].values, index=master.index).rolling(60, min_periods=1).sum()
        return master

    # Convert event times to integer positions on the master index.
    s_pos = idx.searchsorted(starts, side="left")
    e_pos = idx.searchsorted(ends, side="right") - 1

    # Restrict mapped positions to the valid index range.
    s_pos = np.clip(s_pos, 0, len(idx) - 1)
    e_pos = np.clip(e_pos, 0, len(idx) - 1)

    # Count event starts per second.
    np.add.at(master[start_col].values, s_pos, 1)

    # Mark seconds that fall within any event span.
    active = master[active_col].values
    for s, e in zip(s_pos, e_pos):
        if e >= s:
            active[s:e+1] = True
    master[active_col] = active

    # Compute rolling event density over the preceding 60 seconds.
    master[density_col] = pd.Series(master[start_col].values, index=master.index).rolling(60, min_periods=1).sum()

    return master


master = build_master_dataframe(ts_num, ts_cat, freq=CFG["FREQ"])
master = map_event_table(master, event_tables.get("PTT"), prefix="PTT")
master = map_event_table(master, event_tables.get("SBP"), prefix="SBPevt")

print("Master shape:", master.shape)
print("SECTION 2 loaded.")

In [ ]:
# =========================
# SECTION 2A — SANITY CHECKS
# Verify that the master index is strictly increasing, sampled at 1 Hz,
# and that event-mapping outputs were created as expected.
# =========================

assert master.index.is_monotonic_increasing

d = master.index.to_series().diff().dropna()
assert (d == pd.Timedelta(seconds=1)).all(), "Index spacing is not 1 second everywhere"

assert all(c in master.columns for c in ["PTT_events_per_min", "SBPevt_events_per_min"])

print(
    "Event mapping sanity OK:",
    "PTT starts:", int(master["PTT_event_start"].sum()),
    "SBPevt starts:", int(master["SBPevt_event_start"].sum()),
)
print("Master index is 1 Hz and event columns exist.")

In [ ]:
# =========================
# SECTION 2.1 — TRIM TO CORE CHANNEL OVERLAP
# Restrict the master dataframe to the time interval over which all
# core physiological channels are simultaneously available.
# =========================

core_cols = ["SBP", "DBP", "MAP", "HR", "RR", "PTTraw"]

# Confirm that all required core channels are present.
missing = [c for c in core_cols if c not in master.columns]
if missing:
    raise RuntimeError(f"Missing core columns in master: {missing}")

# Determine the common time window shared across all core channels.
core_start = max(master[c].first_valid_index() for c in core_cols)
core_end   = min(master[c].last_valid_index()  for c in core_cols)

if core_start is None or core_end is None or core_end < core_start:
    raise RuntimeError("Core channels do not overlap.")

# Summarise the impact of trimming to the common overlap window.
n_before = len(master)
t_before0, t_before1 = master.index.min(), master.index.max()

master_trimmed = master.loc[core_start:core_end].copy()
n_after = len(master_trimmed)

drop_pct = 100.0 * (n_before - n_after) / max(1, n_before)

print(f"Core window: {core_start} -> {core_end} | seconds: {n_after}")
print(f"Trimmed seconds removed: {n_before - n_after} ({drop_pct:.2f}%)")
print(f"Original window: {t_before0} -> {t_before1}")

# Warn when trimming removes a substantial proportion of the recording.
if drop_pct > 10:
    print("WARNING: >10% of data removed by core overlap trim. Consider relaxed trimming for some analyses.")

master = master_trimmed
print("SECTION 2.1 loaded.")

In [ ]:
# =========================
# SECTION 3 — DERIVED FEATURES
# This section computes signal-derived features used in later QC,
# event classification, and context-aware interpretation.
# It does not depend on QC_status, which is created in Section 4.
# =========================

# --- Core conversion: RR interval to heart rate ---
master["HR_from_RR"] = np.where(master["RR"].notna() & (master["RR"] > 0), 60000.0 / master["RR"], np.nan)

# --- Pre-QC signal consistency features ---
# RR validity based on physiological bounds.
master["RR_invalid"] = (
    master["RR"].isna()
    | (master["RR"] < CFG["RR_MIN_MS"])
    | (master["RR"] > CFG["RR_MAX_MS"])
)

# Sudden beat-to-beat RR changes used as a pre-QC instability flag.
master["RR_jump"] = master["RR"].diff().abs() > CFG["RR_JUMP_MS"]

# Heart-rate mismatch between recorded HR and HR derived from RR interval.
hr_rr_ref = pd.to_numeric(master["HR_from_RR"], errors="coerce").replace(0, np.nan)
master["HR_RR_mismatch"] = (
    master["HR"].notna()
    & hr_rr_ref.notna()
    & (((master["HR"] - hr_rr_ref).abs() / hr_rr_ref) * 100 > CFG["HR_RR_TOL_PCT"])
)

# --- Activity quantiles and motion-state features ---
act = pd.to_numeric(master["Activity"], errors="coerce").dropna()
act_p90 = float(act.quantile(0.90)) if len(act) else np.nan
act_p95 = float(act.quantile(0.95)) if len(act) else np.nan

master["abs_low_motion"] = pd.to_numeric(master["Activity"], errors="coerce") <= CFG["ABS_LOW_MOTION_MAX"]
master["high_motion"] = pd.to_numeric(master["Activity"], errors="coerce") >= max(
    CFG["HIGH_MOTION_FLOOR"], 0 if pd.isna(act_p90) else act_p90
)
master["very_high_motion"] = pd.to_numeric(master["Activity"], errors="coerce") >= max(
    CFG["VHIGH_MOTION_FLOOR"], 0 if pd.isna(act_p95) else act_p95
)

# --- Local SBP baseline and short-term SBP dynamics ---
master["SBP_med120"] = master["SBP"].rolling(
    CFG["BASELINE_LOCAL_SEC"],
    min_periods=CFG["BASELINE_LOCAL_MINP"]
).median()

master["SBP_rise_local"] = master["SBP"] - master["SBP_med120"]
master["SBP_drop_local"] = master["SBP_med120"] - master["SBP"]
master["SBP_diff_1s"] = master["SBP"].diff()

# Heart-rate change over a fixed lag window, used later for supportive physiology.
master["HR_change_10s"] = master["HR"] - master["HR"].shift(CFG["HR_DELTA_SEC"])

# --- Rolling PTT variability feature ---
master["PTTraw_sd"] = master["PTTraw"].rolling(
    CFG["PTT_SD_SEC"],
    min_periods=CFG["PTT_SD_MINP"]
).std()

# --- Robust PTT-noise thresholding without QC_status ---
# A pre-QC clean reference is defined using low motion, RR consistency,
# HR/RR agreement, and absence of sharp SBP transients.
sbp_tmp = pd.to_numeric(master["SBP"], errors="coerce")
abs_d1 = pd.to_numeric(master["SBP_diff_1s"], errors="coerce").abs()

clean_ref = (
    sbp_tmp.notna()
    & master["abs_low_motion"].fillna(False)
    & (~master["RR_invalid"].fillna(False))
    & (~master["RR_jump"].fillna(False))
    & (~master["HR_RR_mismatch"].fillna(False))
    & (abs_d1.fillna(0) < CFG["SBP_SPIKE_MOD"])
)

ptt_sd_clean = pd.to_numeric(master.loc[clean_ref, "PTTraw_sd"], errors="coerce").dropna()

def _robust_thr_from_clean(x: pd.Series, q=0.95, k=6.0) -> float:
    """
    Compute a robust threshold from a reference distribution.

    The threshold is defined as:
        max(quantile(q), median + k * MAD)

    This combines an upper-quantile rule with a robust dispersion-based
    rule to reduce sensitivity to outliers in the reference segment.
    """
    if x is None or len(x) == 0:
        return np.nan
    med = float(x.median())
    mad = float((x - med).abs().median())
    robust = med + k * mad
    qthr = float(x.quantile(q))
    return max(qthr, robust)

ptt_sd_thr_clean = _robust_thr_from_clean(ptt_sd_clean, q=CFG["PTT_SD_Q"], k=6.0)

ptt_sd_all = pd.to_numeric(master["PTTraw_sd"], errors="coerce").dropna()
ptt_sd_thr_all = float(ptt_sd_all.quantile(CFG["PTT_SD_Q"])) if len(ptt_sd_all) else np.nan

# Prefer the clean-reference threshold; use the global distribution only if needed.
ptt_sd_thr = ptt_sd_thr_clean if pd.notna(ptt_sd_thr_clean) else ptt_sd_thr_all

# Event-density threshold using the same clean-reference preference.
if "PTT_events_per_min" in master.columns:
    ptt_evt_clean = pd.to_numeric(master.loc[clean_ref, "PTT_events_per_min"], errors="coerce").dropna()
    ptt_evt_all = pd.to_numeric(master["PTT_events_per_min"], errors="coerce").dropna()
else:
    ptt_evt_clean = pd.Series(dtype=float)
    ptt_evt_all = pd.Series(dtype=float)

ptt_evt_thr_clean = float(ptt_evt_clean.quantile(CFG["PTT_EVT_Q"])) if len(ptt_evt_clean) else np.nan
ptt_evt_thr_all = float(ptt_evt_all.quantile(CFG["PTT_EVT_Q"])) if len(ptt_evt_all) else np.nan
ptt_evt_thr = ptt_evt_thr_clean if pd.notna(ptt_evt_thr_clean) else ptt_evt_thr_all

master["PTT_noisy_sd"] = False if pd.isna(ptt_sd_thr) else (master["PTTraw_sd"] >= ptt_sd_thr)
master["PTT_events_dense"] = False if pd.isna(ptt_evt_thr) else (master["PTT_events_per_min"] >= ptt_evt_thr)

# Combined PTT-noise flag used by downstream QC sections.
master["L2_ptt_noisy"] = master["PTT_noisy_sd"].fillna(False) | master["PTT_events_dense"].fillna(False)

# --- Buffered onset flags used in later sections ---
# These buffered indicators expand transient flags around each second
# to support onset gating in subsequent classification steps.
buf = int(CFG["ONSET_BUF_SEC"])
master["ptt_noisy_buf10"] = (
    master["L2_ptt_noisy"]
    .rolling(2 * buf + 1, center=True, min_periods=1)
    .max()
    .fillna(0)
    .astype(bool)
)
master["high_motion_buf10"] = (
    master["high_motion"]
    .rolling(2 * buf + 1, center=True, min_periods=1)
    .max()
    .fillna(0)
    .astype(bool)
)
master["very_high_motion_buf10"] = (
    master["very_high_motion"]
    .rolling(2 * buf + 1, center=True, min_periods=1)
    .max()
    .fillna(0)
    .astype(bool)
)

print("SECTION 3 loaded.")
print("PTT SD thr:", ptt_sd_thr, "| PTT evt density thr:", ptt_evt_thr)
print("clean_ref seconds:", int(clean_ref.sum()))

In [ ]:
# =========================
# SECTION 3.DIAG — PTT NOISE DIAGNOSTICS
# Summarise the prevalence of PTT-noise flags and their overlap with
# motion-related states, and visualise the PTTraw_sd distribution used
# for threshold selection.
# =========================

print("L2_ptt_noisy seconds:", int(master["L2_ptt_noisy"].sum()))
print("L2_ptt_noisy % of record:", 100 * master["L2_ptt_noisy"].mean())
print("high_motion seconds:", int(master["high_motion"].sum()))
print("very_high_motion seconds:", int(master["very_high_motion"].sum()))

ov1 = (master["L2_ptt_noisy"] & master["high_motion"]).sum()
ov2 = (master["L2_ptt_noisy"] & master["very_high_motion"]).sum()
print("Overlap noisy & high_motion seconds:", int(ov1))
print("Overlap noisy & very_high_motion seconds:", int(ov2))

noisy_total = max(1, int(master["L2_ptt_noisy"].sum()))
print("% of noisy occurring during high_motion:", 100 * ov1 / noisy_total)

import matplotlib.pyplot as plt

ptt_all = pd.to_numeric(master["PTTraw_sd"], errors="coerce").dropna()
ptt_clean = pd.to_numeric(master.loc[clean_ref, "PTTraw_sd"], errors="coerce").dropna()

plt.figure(figsize=(10, 4))
plt.hist(ptt_all, bins=80, alpha=0.35, label="All")
plt.hist(ptt_clean, bins=80, alpha=0.35, label="Clean_ref")
plt.axvline(ptt_sd_thr, color="red", lw=2, label=f"thr={ptt_sd_thr:.2f}")
plt.title("PTTraw_sd distribution (all vs clean_ref)")
plt.legend()
plt.show()

In [ ]:
# =========================
# SECTION 3.1 — CONTEXT STREAMS
# Context labels are used for descriptive interpretation only and are
# not treated as diagnostic evidence.
# =========================

def ffill_str(series: pd.Series, limit_sec: int) -> pd.Series:
    """
    Forward-fill string-based context labels over a bounded interval.

    Because the master dataframe is sampled at 1 Hz, the forward-fill
    limit in seconds is equivalent to the limit in rows.
    """
    s = series.astype("object").replace(["nan", "NaN", "None", ""], np.nan)
    return s.ffill(limit=int(limit_sec))

# Apply conservative forward-filling to Sleep/Wake labels.
# Position labels are allowed a longer carry-forward interval.
SLEEPWAKE_FFILL_SEC = int(min(CFG["CONTEXT_FFILL_LIMIT_SEC"], 2 * 3600))
POSITION_FFILL_SEC  = int(CFG["CONTEXT_FFILL_LIMIT_SEC"])

master["SleepWake_ctx"] = ffill_str(master["SleepWake"], SLEEPWAKE_FFILL_SEC)
master["Position_ctx"]  = ffill_str(master["Position"],  POSITION_FFILL_SEC)

sleep_set = set(CFG["SLEEP_LABELS"])
wake_set  = set(CFG["WAKE_LABELS"])

master["is_sleep_ctx"] = master["SleepWake_ctx"].isin(sleep_set)
master["is_wake_ctx"]  = master["SleepWake_ctx"].isin(wake_set)

sleep_sec = int(master["is_sleep_ctx"].sum())
wake_sec  = int(master["is_wake_ctx"].sum())

# Require both minimum absolute duration and minimum proportional coverage
# before treating Sleep/Wake context as sufficiently represented.
min_sec = int(CFG["SLEEP_MIN_CONTEXT_SEC"])

# Use analyzable time as the denominator when available; otherwise use the full record.
base_den = int(master.get("analyzable", pd.Series(True, index=master.index)).sum())
sleep_cov = sleep_sec / max(1, base_den)
wake_cov  = wake_sec  / max(1, base_den)

MIN_COVERAGE = 0.10

master["sleepwake_context_reliable"] = (
    (sleep_sec >= min_sec) and (wake_sec >= min_sec)
    and (sleep_cov >= MIN_COVERAGE) and (wake_cov >= MIN_COVERAGE)
)

print("SECTION 3.1 loaded.")
print("SleepWake reliable:", bool(master["sleepwake_context_reliable"].iloc[0]))
print(f"Sleep seconds: {sleep_sec} ({sleep_cov*100:.1f}%) | Wake seconds: {wake_sec} ({wake_cov*100:.1f}%)")
print("FFILL caps — SleepWake:", SLEEPWAKE_FFILL_SEC, "sec | Position:", POSITION_FFILL_SEC, "sec")
print("Position uniques:", pd.Series(master["Position_ctx"]).dropna().unique()[:15])

In [ ]:
# =========================
# SECTION 3.2 — POSITION CONFIDENCE AND POSTURE ANCHORS
# Derive position confidence from label recency, motion context, and
# label availability, then identify posture transitions and anchored
# upright windows for downstream OH-related analysis.
# =========================

# -------------------------
# Inputs
# -------------------------
pos_raw = master["Position"].astype("object")
pos_ctx = master["Position_ctx"].astype("object")
raw_present = pos_raw.notna()

# -------------------------
# Position confidence
# Combines label recency, motion penalty, and label availability
# -------------------------
age_sec = np.full(len(master), np.inf, dtype=float)
last_seen = None
for i, t in enumerate(master.index):
    if bool(raw_present.iloc[i]):
        last_seen = t
        age_sec[i] = 0.0
    elif last_seen is not None:
        age_sec[i] = (t - last_seen).total_seconds()
master["pos_age_sec"] = age_sec

tau = float(CFG["POS_TAU_SEC"])
freshness_conf = np.clip(np.exp(-master["pos_age_sec"] / tau), 0, 1)

motion_penalty = 0.5
motion_factor = np.where(
    master["high_motion_buf10"].fillna(False).astype(bool),
    (1 - motion_penalty),
    1.0
)

known_factor = np.where(pos_ctx.notna(), 1.0, 0.0)
master["pos_conf"] = (freshness_conf * motion_factor * known_factor).astype(float)

# -------------------------
# Raw posture masks derived from the context stream
# -------------------------
SUPINE = set(CFG["SUPINE_LABELS"])
UPRIGHT = set(CFG["UPRIGHT_LABELS"])

pos_supine = pos_ctx.isin(SUPINE).fillna(False).astype(bool)
pos_upright = pos_ctx.isin(UPRIGHT).fillna(False).astype(bool)

master["pos_supine"] = pos_supine
master["pos_upright"] = pos_upright

# Apply a confidence threshold to define reliable posture labels.
conf_min = float(CFG["POS_CONF_MIN_FOR_SUPINE_LABEL"])
pos_supine_rel = (pos_supine & (master["pos_conf"] >= conf_min)).astype(bool)
pos_upright_rel = (pos_upright & (master["pos_conf"] >= conf_min)).astype(bool)

master["pos_supine_reliable"] = pos_supine_rel
master["pos_upright_reliable"] = pos_upright_rel

# -------------------------
# Stability modelling to reduce chatter in posture labels
# -------------------------
# Rolling posture proportions are used to define a prevailing state.
POS_WIN = 15
MIN_STABLE_PCT = 0.70
STATE_GAP_FILL_SEC = 300
REFRACTORY_SEC = 60

sup_p = pos_supine_rel.rolling(POS_WIN, min_periods=POS_WIN).mean()
upr_p = pos_upright_rel.rolling(POS_WIN, min_periods=POS_WIN).mean()

max_sup = float(sup_p.max(skipna=True)) if sup_p.notna().any() else np.nan
max_upr = float(upr_p.max(skipna=True)) if upr_p.notna().any() else np.nan

pct_sup_ge_07 = float((sup_p >= 0.7).mean(skipna=True) * 100) if sup_p.notna().any() else np.nan
pct_upr_ge_07 = float((upr_p >= 0.7).mean(skipna=True) * 100) if upr_p.notna().any() else np.nan

# Define the prevailing posture state per second:
# - "supine" when the rolling supine proportion exceeds the threshold
# - "upright" when the rolling upright proportion exceeds the threshold
#   and supine is not simultaneously stable
# - otherwise "unknown"
prev_state = pd.Series("unknown", index=master.index, dtype="object")
prev_state.loc[sup_p >= MIN_STABLE_PCT] = "supine"
prev_state.loc[(upr_p >= MIN_STABLE_PCT) & ~(sup_p >= MIN_STABLE_PCT)] = "upright"

master["pos_state_prevailing_raw"] = prev_state

# Fill short unknown gaps to preserve posture transitions across brief interruptions.
prev_state_ff = prev_state.replace("unknown", np.nan).ffill(limit=STATE_GAP_FILL_SEC)
master["pos_state_prevailing_ff"] = prev_state_ff

# Candidate posture transitions are defined as supine-to-upright changes
# after gap filling.
cand_transition = (prev_state_ff.shift(1, fill_value=np.nan) == "supine") & (prev_state_ff == "upright")

# Apply a refractory period to avoid repeated counting of rapid label chatter.
transition_times = master.index[cand_transition]
keep = []
last_t = None
for t in transition_times:
    if last_t is None or (t - last_t).total_seconds() >= REFRACTORY_SEC:
        keep.append(t)
        last_t = t
transition_times = pd.DatetimeIndex(keep)

master["pos_transition_supine_to_upright"] = master.index.isin(transition_times)

# -------------------------
# Anchor strength and posture window definition
# -------------------------
PRE = int(CFG["ANCHOR_PRE_SEC"])
POST = int(CFG["ANCHOR_POST_SEC"])
ANCHOR_THR = float(CFG["ANCHOR_THR"])

anchor_strength = pd.Series(0.0, index=master.index)
for t in transition_times:
    pre = master.loc[t - pd.Timedelta(seconds=PRE):t, "pos_conf"]
    post = master.loc[t:t + pd.Timedelta(seconds=POST), "pos_conf"]
    anchor_strength.loc[t] = float(min(
        pre.mean() if len(pre) else 0.0,
        post.mean() if len(post) else 0.0
    ))
master["pos_anchor_strength"] = anchor_strength

OH_WINDOW = int(CFG["OH_WINDOW_SEC"])
posture_window = pd.Series(False, index=master.index)
credible_transition_times = []
for t in transition_times:
    if float(master.loc[t, "pos_anchor_strength"]) >= ANCHOR_THR:
        posture_window.loc[t: t + pd.Timedelta(seconds=OH_WINDOW)] = True
        credible_transition_times.append(t)
master["posture_window_after_upright"] = posture_window.astype(bool)

# -------------------------
# Summary reporting table
# -------------------------
raw_transitions = int((pos_supine.shift(1, fill_value=False) & pos_upright).sum())

pos_meta = pd.DataFrame([{
    "pos_known_pct": float(pos_ctx.notna().mean() * 100),
    "n_transitions_raw_supine_to_upright": raw_transitions,
    "n_transitions_debounced": int(master["pos_transition_supine_to_upright"].sum()),
    "n_credible_transitions": int(len(credible_transition_times)),
    "posture_window_seconds": int(master["posture_window_after_upright"].sum()),
    "median_pos_conf": float(master["pos_conf"].median()) if master["pos_conf"].notna().any() else np.nan,

    # Stability diagnostics
    "POS_WIN": POS_WIN,
    "MIN_STABLE_PCT": MIN_STABLE_PCT,
    "STATE_GAP_FILL_SEC": STATE_GAP_FILL_SEC,
    "REFRACTORY_SEC": REFRACTORY_SEC,
    "max_supine_prop": max_sup,
    "max_upright_prop": max_upr,
    "%windows_sup>=0.7": pct_sup_ge_07,
    "%windows_upr>=0.7": pct_upr_ge_07,

    "ANCHOR_THR": ANCHOR_THR,
}])

print("SECTION 3.2 loaded.")
display(pos_meta)

In [ ]:
# =========================
# SECTION 3.2.DIAG — POSITION STABILITY DIAGNOSTIC
# Evaluate whether reliable posture labels form sufficiently stable
# windows to support transition detection.
# =========================

POS_WIN = 15

sup = master["pos_supine_reliable"].fillna(False).astype(bool)
upr = master["pos_upright_reliable"].fillna(False).astype(bool)

sup_p = sup.rolling(POS_WIN, min_periods=POS_WIN).mean()
upr_p = upr.rolling(POS_WIN, min_periods=POS_WIN).mean()

print("Max supine proportion in any 15s window:", float(sup_p.max(skipna=True)))
print("Max upright proportion in any 15s window:", float(upr_p.max(skipna=True)))

# Proportion of windows exceeding candidate stability thresholds
print("% windows sup>=0.6:", float((sup_p >= 0.6).mean(skipna=True)) * 100)
print("% windows upr>=0.6:", float((upr_p >= 0.6).mean(skipna=True)) * 100)
print("% windows sup>=0.7:", float((sup_p >= 0.7).mean(skipna=True)) * 100)
print("% windows upr>=0.7:", float((upr_p >= 0.7).mean(skipna=True)) * 100)

In [ ]:
# =========================
# SECTION 4 — QC ENGINE (VALID / Intermediate-Quality / ARTEFACT)
# Classify each second according to signal-quality evidence from
# rhythm consistency, PTT noise, motion, and abrupt SBP dynamics.
# =========================

def _b(s):
    """
    Coerce an input series to a boolean series aligned to the master index.
    Missing values are treated as False.
    """
    return pd.Series(s, index=master.index).fillna(False).astype(bool)

master["L1_fail"] = _b(master["RR_invalid"]) | _b(master["RR_jump"]) | _b(master["HR_RR_mismatch"])
master["L2_fail"] = _b(master["L2_ptt_noisy"])
master["L3_high"] = _b(master["high_motion"])
master["L3_vhigh"] = _b(master["very_high_motion"])

abs_d1 = pd.to_numeric(master["SBP_diff_1s"], errors="coerce").abs()

master["L4_spike_hard"] = (abs_d1 > CFG["SBP_SPIKE_HARD"]).fillna(False).astype(bool)
master["L4_spike_mod"]  = ((abs_d1 >= CFG["SBP_SPIKE_MOD"]) & (abs_d1 < CFG["SBP_SPIKE_HARD"])).fillna(False).astype(bool)

# --- Default QC state ---
master["QC_status"] = "valid"
master["QC_reason"] = "OK"

mask_l1 = _b(master["L1_fail"])
mask_l2_motion = _b(master["L2_fail"]) & _b(master["L3_high"])
mask_l4_corr = _b(master["L4_spike_hard"]) & (mask_l1 | _b(master["L2_fail"]) | _b(master["L3_vhigh"]))

# --- Artefact rules ---
# These rules identify strongly corrupted seconds that are excluded
# from analyzable output.
artefact = mask_l1 | mask_l2_motion | mask_l4_corr
master.loc[artefact, "QC_status"] = "artefact"

# Deterministic assignment of the primary artefact reason.
master.loc[mask_l1, "QC_reason"] = "L1_RRHR_FAIL"
master.loc[(~mask_l1) & mask_l2_motion & (master["QC_status"] == "artefact"), "QC_reason"] = "L2_PTT_PLUS_MOTION"
master.loc[(~mask_l1) & (~mask_l2_motion) & mask_l4_corr & (master["QC_status"] == "artefact"), "QC_reason"] = "L4_BP_DYNAMICS"

# --- Intermediate-Quality rules ---
# Intermediate-Quality seconds are retained as analyzable but marked as uncertain.
# Intermediate-Quality classification is only allowed in the absence of L1 failure
# and high-motion conditions.
intermediate_quality = (
    (master["QC_status"] == "valid")
    & (~mask_l1)
    & (~_b(master["L3_high"]))
    & (
        _b(master["L2_fail"])
        | _b(master["L4_spike_mod"])
        | (_b(master["L4_spike_hard"]) & (~mask_l4_corr))
    )
)

master.loc[intermediate_quality, "QC_status"] = "Intermediate-Quality"
master.loc[intermediate_quality & _b(master["L2_fail"]), "QC_reason"] = "L2_PTT_Intermediate-Quality"
master.loc[
    intermediate_quality
    & (~_b(master["L2_fail"]))
    & (_b(master["L4_spike_mod"]) | _b(master["L4_spike_hard"])),
    "QC_reason"
] = "L4_BP_UNSTABLE_Intermediate-Quality"

# --- Analyzable mask ---
# Valid and Intermediate-Quality seconds are retained for downstream analysis.
master["analyzable"] = master["QC_status"].isin(["valid", "Intermediate-Quality"])

# --- Summary outputs ---
qc_counts = master["QC_status"].value_counts()
qc_pct = (qc_counts / len(master) * 100).round(2)

print("QC summary:")
display(pd.concat([qc_counts.rename("count"), qc_pct.rename("pct")], axis=1))

print("\nQC reason breakdown (top 10):")
display(master["QC_reason"].value_counts().head(10).rename_axis("QC_reason").reset_index(name="count"))

print("SECTION 4 loaded.")

In [ ]:
# =========================
# SECTION 4.1 — QC INVARIANTS
# Verify that QC outputs satisfy the locked classification rules.
# These assertions act as internal consistency checks and do not alter
# the analysis pipeline.
# =========================

assert set(master["QC_status"].dropna().unique()).issubset({"valid", "Intermediate-Quality", "artefact"})
assert (master.loc[master["L1_fail"].fillna(False), "QC_status"] == "artefact").all()

artefact_mask = master["QC_status"].eq("artefact")
mask_l1 = master["L1_fail"].fillna(False)
mask_l2_motion = master["L2_fail"].fillna(False) & master["L3_high"].fillna(False)
mask_l4_corr = master["L4_spike_hard"].fillna(False) & (
    master["L1_fail"].fillna(False)
    | master["L2_fail"].fillna(False)
    | master["L3_vhigh"].fillna(False)
)
assert ((mask_l1 | mask_l2_motion | mask_l4_corr).loc[artefact_mask]).all()

intermediate_quality_mask = master["QC_status"].eq("Intermediate-Quality")
assert (~master.loc[intermediate_quality_mask, "L1_fail"].fillna(False)).all()
assert (~master.loc[intermediate_quality_mask, "L3_high"].fillna(False)).all()

print("QC invariants passed.")

In [ ]:
# =========================
# SECTION 4.15 — L4 TRIGGER GUARD
# Prevent short-lived SBP glitches or outliers from triggering event
# thresholds. This section does not modify QC_status; it only defines
# analyzable_for_threshold for threshold-based event detection.
# =========================

sbp = pd.to_numeric(master["SBP"], errors="coerce")

# Restrict trigger-guard evaluation to seconds already considered analyzable
# under the QC framework and with non-missing SBP data.
analyzable = master["analyzable"].fillna(False).astype(bool) & sbp.notna()

# -------------------------
# Guard 1: jump-and-return pattern over a multi-second interval
# -------------------------
# Detect abrupt SBP changes relative to 5 seconds earlier.
JUMP_SEC = 5
JUMP_THR = 18.0

# Define return over the subsequent window as re-entering a tolerance band
# around the pre-jump SBP level.
RETURN_WIN_SEC = 10
RETURN_TOL = float(CFG.get("SPIKE_GLITCH_RETURN_TOL", 5))

pre_val = sbp.shift(JUMP_SEC)
jump_5s = (sbp - pre_val).abs() >= JUMP_THR

# Future min/max values over the return window.
future_min = sbp.rolling(RETURN_WIN_SEC, min_periods=1).min().shift(-(RETURN_WIN_SEC - 1))
future_max = sbp.rolling(RETURN_WIN_SEC, min_periods=1).max().shift(-(RETURN_WIN_SEC - 1))

returned = (
    pre_val.notna() & future_min.notna() & future_max.notna()
    & (pre_val >= (future_min - RETURN_TOL))
    & (pre_val <= (future_max + RETURN_TOL))
)

glitch_jump_return = analyzable & jump_5s.fillna(False) & returned.fillna(False)

# -------------------------
# Guard 2: Hampel-like local outlier rule
# -------------------------
# Identify isolated SBP outliers relative to local behaviour using a
# rolling median and rolling median absolute deviation.
MAD_WIN = 31
MAD_K = 6.0

med = sbp.rolling(MAD_WIN, min_periods=max(10, MAD_WIN // 3)).median()
mad = (sbp - med).abs().rolling(MAD_WIN, min_periods=max(10, MAD_WIN // 3)).median()

# Add a small constant to avoid division by zero in near-flat segments.
eps = 1e-6
z = (sbp - med).abs() / (mad + eps)
outlier_mad = analyzable & z.notna() & (z >= MAD_K)

# -------------------------
# Combined trigger guard
# -------------------------
master["L4_trigger_guard"] = (glitch_jump_return | outlier_mad).fillna(False).astype(bool)

# QC remains unchanged; only threshold-triggering eligibility is modified.
master["analyzable_for_threshold"] = analyzable & (~master["L4_trigger_guard"])

# -------------------------
# Validation output
# -------------------------
print("SECTION 4.15 loaded.")
print("Glitch jump-return seconds excluded:", int(glitch_jump_return.sum()))
print("MAD outlier seconds excluded:", int(outlier_mad.sum()))
print("Total L4_trigger_guard seconds excluded:", int(master["L4_trigger_guard"].sum()))
print("Analyzable seconds (QC):", int(analyzable.sum()))
print("Analyzable seconds for threshold detection:", int(master["analyzable_for_threshold"].sum()))

# Summarise exclusions across QC categories.
if "QC_status" in master.columns:
    excl = master["L4_trigger_guard"].fillna(False)
    print("\nL4_trigger_guard overlap by QC_status:")
    display(master.loc[excl, "QC_status"].value_counts().rename_axis("QC_status").reset_index(name="count"))

In [ ]:
# =========================
# SECTION 4.2 — PROTOCOL EXTRACTION (SUPINE, SEATED, AND DRIFT)
# Multi-day resolution of protocol clock-times with extraction of
# local summary windows around matched timestamps.
# =========================

PROTOCOL_TOL_MIN = 2
ROWS_EACH_SIDE = 5
QC_MODE = "valid_plus_Intermediate-Quality"  # or "valid_only"

# -------------------------
# Helper functions
# -------------------------
def _select_best_match(index: pd.DatetimeIndex, hms: str, tol_min: int, prefer: str = "first"):
    """
    Resolve a clock-time string (HH:MM:SS) to the nearest matching
    timestamp in a multi-day index within a specified tolerance.
    """
    tol = pd.Timedelta(minutes=tol_min)

    # First, search for an exact HH:MM:SS match across any day.
    matches = index[index.strftime("%H:%M:%S") == hms]
    if len(matches) > 0:
        chosen = matches[0] if prefer == "first" else matches[-1]
        return chosen, pd.Timedelta(seconds=0)

    # Otherwise, search for the nearest match across all unique dates.
    unique_dates = pd.Index(index.normalize().unique())
    best = pd.NaT
    best_diff = None

    for d in unique_dates:
        target = pd.Timestamp(str(d.date()) + " " + hms)
        pos = index.get_indexer([target], method="nearest")[0]
        cand = index[pos]
        diff = abs(cand - target)
        if best_diff is None or diff < best_diff:
            best = cand
            best_diff = diff

    if best_diff is not None and best_diff <= tol:
        return best, best_diff

    return pd.NaT, pd.NaT


def resolve_hms_list(index: pd.DatetimeIndex, times_hms: list, tol_min: int, prefer: str):
    """
    Resolve a list of clock-time strings to timestamps in the recording index.
    Returns both resolved timestamps and their absolute matching differences.
    """
    if times_hms is None or len(times_hms) == 0:
        return [], []

    resolved = []
    diffs = []
    for t in times_hms:
        rt, diff = _select_best_match(index, t, tol_min=tol_min, prefer=prefer)
        resolved.append(rt)
        diffs.append(diff)
    return resolved, diffs


def extract_protocol_points(master: pd.DataFrame, resolved_times: list, diffs: list, label: str) -> pd.DataFrame:
    """
    Extract local windows around resolved protocol timestamps and summarise
    median physiological values within each window.
    """
    out_cols = [
        "condition", "resolved_time", "abs_diff_sec", "window_start", "window_end",
        "SBP_med", "DBP_med", "MAP_med", "HR_med", "PTTraw_med",
        "n_SBP", "n_DBP", "n_MAP", "n_HR", "n_PTTraw",
        "n_valid", "n_Intermediate-Quality", "n_artefact", "note"
    ]

    cols = [c for c in ["SBP", "DBP", "MAP", "HR", "PTTraw", "QC_status"] if c in master.columns]
    idx = master.index
    rows = []

    # Return an empty table with a consistent schema if no timestamps were resolved.
    if resolved_times is None or len(resolved_times) == 0:
        return pd.DataFrame(columns=out_cols)

    for t, diff in zip(pd.to_datetime(resolved_times), diffs):
        if pd.isna(t):
            rows.append({
                "condition": label,
                "resolved_time": pd.NaT,
                "abs_diff_sec": np.nan,
                "window_start": pd.NaT,
                "window_end": pd.NaT,
                "SBP_med": np.nan, "DBP_med": np.nan, "MAP_med": np.nan, "HR_med": np.nan, "PTTraw_med": np.nan,
                "n_SBP": 0, "n_DBP": 0, "n_MAP": 0, "n_HR": 0, "n_PTTraw": 0,
                "n_valid": 0, "n_Intermediate-Quality": 0, "n_artefact": 0,
                "note": "no_match_within_tolerance"
            })
            continue

        pos = idx.get_indexer([t], method="nearest")[0]
        lo = max(0, pos - ROWS_EACH_SIDE)
        hi = min(len(idx), pos + ROWS_EACH_SIDE + 1)
        win_idx = idx[lo:hi]

        seg = master.loc[win_idx, cols].copy()

        # Restrict to the requested QC subset if QC status is available.
        if "QC_status" in seg.columns:
            if QC_MODE == "valid_only":
                seg_use = seg[seg["QC_status"] == "valid"]
            else:
                seg_use = seg[seg["QC_status"].isin(["valid", "Intermediate-Quality"])]
        else:
            seg_use = seg.copy()

        # Count usable observations for each variable.
        n_SBP = int(pd.to_numeric(seg_use.get("SBP", np.nan), errors="coerce").notna().sum())
        n_DBP = int(pd.to_numeric(seg_use.get("DBP", np.nan), errors="coerce").notna().sum())
        n_MAP = int(pd.to_numeric(seg_use.get("MAP", np.nan), errors="coerce").notna().sum())
        n_HR  = int(pd.to_numeric(seg_use.get("HR", np.nan), errors="coerce").notna().sum())
        n_PTT = int(pd.to_numeric(seg_use.get("PTTraw", np.nan), errors="coerce").notna().sum())

        SBP_med = float(pd.to_numeric(seg_use.get("SBP", np.nan), errors="coerce").median()) if n_SBP else np.nan
        DBP_med = float(pd.to_numeric(seg_use.get("DBP", np.nan), errors="coerce").median()) if n_DBP else np.nan
        MAP_med = float(pd.to_numeric(seg_use.get("MAP", np.nan), errors="coerce").median()) if n_MAP else np.nan
        HR_med  = float(pd.to_numeric(seg_use.get("HR", np.nan), errors="coerce").median()) if n_HR else np.nan
        PTT_med = float(pd.to_numeric(seg_use.get("PTTraw", np.nan), errors="coerce").median()) if n_PTT else np.nan

        note = "ok"
        if (n_SBP == 0 and n_DBP == 0 and n_MAP == 0):
            note = "missing_core_values"

        rows.append({
            "condition": label,
            "resolved_time": idx[pos],
            "abs_diff_sec": float(diff.total_seconds()) if isinstance(diff, pd.Timedelta) else np.nan,
            "window_start": win_idx.min(),
            "window_end": win_idx.max(),
            "SBP_med": SBP_med, "DBP_med": DBP_med, "MAP_med": MAP_med, "HR_med": HR_med, "PTTraw_med": PTT_med,
            "n_SBP": n_SBP, "n_DBP": n_DBP, "n_MAP": n_MAP, "n_HR": n_HR, "n_PTTraw": n_PTT,
            "n_valid": int((seg.get("QC_status", "valid") == "valid").sum()) if "QC_status" in seg.columns else np.nan,
            "n_Intermediate-Quality": int((seg.get("QC_status", "valid") == "Intermediate-Quality").sum()) if "QC_status" in seg.columns else np.nan,
            "n_artefact": int((seg.get("QC_status", "valid") == "artefact").sum()) if "QC_status" in seg.columns else np.nan,
            "note": note
        })

    return pd.DataFrame(rows, columns=out_cols)


def safe_median_col(df: pd.DataFrame, col: str) -> float:
    """
    Return the median of a column, or NaN if the dataframe is empty
    or the column is unavailable.
    """
    if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
        return np.nan
    if col not in df.columns:
        return np.nan
    s = pd.to_numeric(df[col], errors="coerce")
    return float(s.median()) if s.notna().any() else np.nan


# -------------------------
# Resolve protocol timestamps
# -------------------------
supine_dt, supine_diff = resolve_hms_list(master.index, supine_times, tol_min=PROTOCOL_TOL_MIN, prefer="first")
seated_dt, seated_diff = resolve_hms_list(master.index, seated_times, tol_min=PROTOCOL_TOL_MIN, prefer="first")
drift_dt,  drift_diff  = resolve_hms_list(master.index, drift_times,  tol_min=PROTOCOL_TOL_MIN, prefer="last")

# -------------------------
# Extract protocol tables
# -------------------------
protocol_supine_tbl = extract_protocol_points(master, supine_dt, supine_diff, "Supine")
protocol_seated_tbl = extract_protocol_points(master, seated_dt, seated_diff, "Seated")
protocol_drift_tbl  = extract_protocol_points(master, drift_dt,  drift_diff,  "Drift")

# Combine all protocol extracts into a single table.
protocol_tbl = pd.concat([protocol_supine_tbl, protocol_seated_tbl, protocol_drift_tbl], ignore_index=True)

# -------------------------
# Compute protocol-derived baselines
# -------------------------
baseline_protocol_SBP_supine = safe_median_col(protocol_supine_tbl, "SBP_med")
baseline_protocol_SBP_seated = safe_median_col(protocol_seated_tbl, "SBP_med")

master["baseline_protocol_SBP_supine"] = baseline_protocol_SBP_supine
master["baseline_protocol_SBP_seated"] = baseline_protocol_SBP_seated

if pd.isna(baseline_protocol_SBP_seated):
    print("Note: no seated protocol baseline available.")

# -------------------------
# Summarise reporting windows
# -------------------------
protocol_windows_supine = [(protocol_supine_tbl["window_start"].min(), protocol_supine_tbl["window_end"].max())] \
    if (len(protocol_supine_tbl) and protocol_supine_tbl["window_start"].notna().any()) else []

protocol_windows_seated = [(protocol_seated_tbl["window_start"].min(), protocol_seated_tbl["window_end"].max())] \
    if (len(protocol_seated_tbl) and protocol_seated_tbl["window_start"].notna().any()) else []

protocol_windows_drift = [(protocol_drift_tbl["window_start"].min(), protocol_drift_tbl["window_end"].max())] \
    if (len(protocol_drift_tbl) and protocol_drift_tbl["window_start"].notna().any()) else []

protocol_summary = pd.DataFrame([{
    "patient_id": patient_id,
    "protocol_tol_min": PROTOCOL_TOL_MIN,
    "QC_MODE": QC_MODE,
    "baseline_protocol_SBP_supine": baseline_protocol_SBP_supine,
    "baseline_protocol_SBP_seated": baseline_protocol_SBP_seated,
    "n_supine_points_matched": int(protocol_supine_tbl["resolved_time"].notna().sum()) if len(protocol_supine_tbl) else 0,
    "n_seated_points_matched": int(protocol_seated_tbl["resolved_time"].notna().sum()) if len(protocol_seated_tbl) else 0,
    "n_drift_points_matched": int(protocol_drift_tbl["resolved_time"].notna().sum()) if len(protocol_drift_tbl) else 0,
    "supine_window_start": protocol_windows_supine[0][0] if protocol_windows_supine else pd.NaT,
    "supine_window_end": protocol_windows_supine[0][1] if protocol_windows_supine else pd.NaT,
    "seated_window_start": protocol_windows_seated[0][0] if protocol_windows_seated else pd.NaT,
    "seated_window_end": protocol_windows_seated[0][1] if protocol_windows_seated else pd.NaT,
    "drift_window_start": protocol_windows_drift[0][0] if protocol_windows_drift else pd.NaT,
    "drift_window_end": protocol_windows_drift[0][1] if protocol_windows_drift else pd.NaT,
}])

print("SECTION 4.2 loaded.")
display(protocol_summary)
display(protocol_tbl.head(40))

In [ ]:
# =========================
# SECTION 5 — MOTION AND EFFORT CONTEXT
# Derive broad motion context and narrower effort-likely context for descriptive interpretation of blood-pressure changes.
# =========================

ACT_BOUT_MIN_SEC = 15
ACT_GAP_BRIDGE_SEC = 5
ACT_RECOVERY_SEC = 600  # 10-minute recovery after motion bouts

HR_DELTA_WINDOW_SEC = 60      # Rolling baseline window for HR change
HR_DELTA_THR_BPM = 5          # Minimum HR rise above rolling baseline
HR_BOUT_MIN_SEC = 30          # Minimum duration for HR corroboration
EFFORT_RECOVERY_SEC = 300     # Recovery period after effort-likely bouts

def bridge_short_false_gaps(mask: pd.Series, max_gap_sec: int) -> pd.Series:
    """
    Fill short False gaps within a boolean mask to preserve continuity
    across brief interruptions.
    """
    s = pd.Series(mask, index=master.index).fillna(False).astype(bool).copy()
    if s.sum() == 0 or max_gap_sec <= 0:
        return s

    grp = s.ne(s.shift()).cumsum()
    blocks = []
    for _, block in s.groupby(grp):
        blocks.append({
            "val": bool(block.iloc[0]),
            "start": block.index[0],
            "end": block.index[-1],
            "len": len(block)
        })

    for i in range(1, len(blocks) - 1):
        prev_b, cur_b, next_b = blocks[i - 1], blocks[i], blocks[i + 1]
        if (not cur_b["val"]) and prev_b["val"] and next_b["val"] and cur_b["len"] <= max_gap_sec:
            s.loc[cur_b["start"]:cur_b["end"]] = True
    return s


def extract_runs(mask: pd.Series, min_len_sec: int) -> pd.DataFrame:
    """
    Extract contiguous True runs from a boolean mask and retain only
    those meeting the minimum duration criterion.
    """
    s = pd.Series(mask, index=master.index).fillna(False).astype(bool)
    if not s.any():
        return pd.DataFrame(columns=["start", "end", "duration_sec"])

    gid = s.ne(s.shift()).cumsum()
    rows = []
    for _, block in s.groupby(gid):
        if not bool(block.iloc[0]):
            continue
        start, end = block.index[0], block.index[-1]
        dur = int((end - start).total_seconds()) + 1
        if dur >= min_len_sec:
            rows.append({"start": start, "end": end, "duration_sec": dur})
    return pd.DataFrame(rows)


# Use analyzable_for_threshold when available so that effort-related
# HR rises are evaluated consistently with the trigger-guard logic.
an_thr = master.get(
    "analyzable_for_threshold",
    master.get("analyzable", pd.Series(False, index=master.index))
).fillna(False).astype(bool)

# -------------------------
# (A) Motion context
# Broad movement-related context based on sustained high or very-high motion,
# extended by a recovery window.
# -------------------------
motion_raw = master["high_motion"].fillna(False).astype(bool) | master["very_high_motion"].fillna(False).astype(bool)
motion_mask = bridge_short_false_gaps(motion_raw, ACT_GAP_BRIDGE_SEC)
motion_bouts = extract_runs(motion_mask, ACT_BOUT_MIN_SEC)

motion_context_mask = pd.Series(False, index=master.index)
for _, b in motion_bouts.iterrows():
    s = pd.to_datetime(b["start"])
    e = pd.to_datetime(b["end"])
    motion_context_mask.loc[s: e + pd.Timedelta(seconds=ACT_RECOVERY_SEC)] = True

# -------------------------
# (B) Effort-likely context
# Narrower context requiring both motion and HR rise above a rolling baseline.
# -------------------------
hr = pd.to_numeric(master["HR"], errors="coerce")
hr_base = hr.rolling(HR_DELTA_WINDOW_SEC, min_periods=max(10, HR_DELTA_WINDOW_SEC // 3)).median()
hr_delta = hr - hr_base

hr_rise = an_thr & hr.notna() & hr_base.notna() & (hr_delta >= HR_DELTA_THR_BPM)
hr_rise = bridge_short_false_gaps(hr_rise, ACT_GAP_BRIDGE_SEC)

# Constrain HR-rise evidence to periods within the motion context.
hr_rise_in_motion = hr_rise & motion_context_mask.fillna(False)

hr_bouts = extract_runs(hr_rise_in_motion, HR_BOUT_MIN_SEC)

effort_likely_mask = pd.Series(False, index=master.index)
for _, b in hr_bouts.iterrows():
    s = pd.to_datetime(b["start"])
    e = pd.to_datetime(b["end"])
    effort_likely_mask.loc[s: e + pd.Timedelta(seconds=EFFORT_RECOVERY_SEC)] = True

# -------------------------
# Outputs
# -------------------------
master["motion_context_mask"] = motion_context_mask.fillna(False).astype(bool)
master["effort_likely_mask"] = effort_likely_mask.fillna(False).astype(bool)

# Maintain effort_context_mask as the broad motion-context indicator
# for backwards compatibility with downstream sections.
master["effort_context_mask"] = master["motion_context_mask"]

effort_meta_tbl = pd.DataFrame([{
    "patient_id": patient_id,
    "motion_context_seconds": int(master["motion_context_mask"].sum()),
    "effort_likely_seconds": int(master["effort_likely_mask"].sum()),
    "motion_bouts_n": int(len(motion_bouts)),
    "hr_bouts_n": int(len(hr_bouts)),
    "ACT_BOUT_MIN_SEC": ACT_BOUT_MIN_SEC,
    "ACT_RECOVERY_SEC": ACT_RECOVERY_SEC,
    "HR_DELTA_WINDOW_SEC": HR_DELTA_WINDOW_SEC,
    "HR_DELTA_THR_BPM": HR_DELTA_THR_BPM,
    "HR_BOUT_MIN_SEC": HR_BOUT_MIN_SEC,
    "EFFORT_RECOVERY_SEC": EFFORT_RECOVERY_SEC,
}])

print("SECTION 5 loaded.")
print("motion_context_mask seconds:", int(master["motion_context_mask"].sum()))
print("effort_likely_mask seconds:", int(master["effort_likely_mask"].sum()))
display(effort_meta_tbl)

In [ ]:
# =========================
# SECTION 5.1 — BASELINE MODEL
# Derive candidate SBP baselines using protocol-based and resting-state approaches, then select the active reference baseline according to CFG["BASELINE_ACTIVE_METHOD"].
# =========================

def median_if_any(x: pd.Series) -> float:
    """
    Return the median of a numeric series, or NaN if no valid values exist.
    """
    s = pd.to_numeric(x, errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan

# Prefer analyzable_for_threshold when available so that baseline
# estimation respects the L4 trigger-guard.
sbp = pd.to_numeric(master["SBP"], errors="coerce")
an_thr = master.get("analyzable_for_threshold", master["analyzable"].fillna(False)).fillna(False).astype(bool) & sbp.notna()

# Define a resting analyzable mask by excluding broad motion-context periods.
resting_analyzable = an_thr & (~master["effort_context_mask"].fillna(False).astype(bool))

# Protocol-derived baseline from Section 4.2.
baseline_protocol = np.nan
if "baseline_protocol_SBP_supine" in master.columns and master["baseline_protocol_SBP_supine"].notna().any():
    baseline_protocol = float(master["baseline_protocol_SBP_supine"].dropna().iloc[0])

# Stable clean resting baseline:
# resting analyzable + no buffered PTT noise + no buffered high motion + no hard SBP spikes
stable_clean_resting = (
    resting_analyzable
    & (~master["ptt_noisy_buf10"].fillna(False).astype(bool))
    & (~master["high_motion_buf10"].fillna(False).astype(bool))
    & (~master["L4_spike_hard"].fillna(False).astype(bool))
)

baseline_stable = median_if_any(master.loc[stable_clean_resting, "SBP"])

# Low-motion clean resting baseline:
# resting analyzable + absolute low motion + no buffered PTT noise
low_motion_clean_resting = (
    resting_analyzable
    & master["abs_low_motion"].fillna(False).astype(bool)
    & (~master["ptt_noisy_buf10"].fillna(False).astype(bool))
)

baseline_low_motion = median_if_any(master.loc[low_motion_clean_resting, "SBP"])

# Global resting analyzable baseline:
# all resting analyzable seconds without additional signal restrictions
baseline_global = median_if_any(master.loc[resting_analyzable, "SBP"])

baseline_candidates = pd.DataFrame([
    {"source": "protocol_supine",           "baseline_SBP": baseline_protocol,   "seconds_used": np.nan},
    {"source": "stable_clean_resting",      "baseline_SBP": baseline_stable,     "seconds_used": int(stable_clean_resting.sum())},
    {"source": "low_motion_clean_resting",  "baseline_SBP": baseline_low_motion, "seconds_used": int(low_motion_clean_resting.sum())},
    {"source": "global_resting_analyzable", "baseline_SBP": baseline_global,     "seconds_used": int(resting_analyzable.sum())},
])

# Select the reference baseline according to the configured active method.
active = CFG.get("BASELINE_ACTIVE_METHOD", "protocol_supine")

baseline_ref_SBP = np.nan
baseline_ref_source = None

# Try the active method first.
hit = baseline_candidates.loc[baseline_candidates["source"].eq(active)]
if len(hit) and pd.notna(hit["baseline_SBP"].iloc[0]):
    baseline_ref_SBP = float(hit["baseline_SBP"].iloc[0])
    baseline_ref_source = str(active)
else:
    # Fallback to the first available non-missing candidate.
    for _, r in baseline_candidates.iterrows():
        if pd.notna(r["baseline_SBP"]):
            baseline_ref_SBP = float(r["baseline_SBP"])
            baseline_ref_source = str(r["source"])
            break

master["baseline_ref_SBP"] = baseline_ref_SBP
master["baseline_ref_source"] = baseline_ref_source

baseline_model_tbl = pd.DataFrame([{
    "patient_id": patient_id,
    "baseline_ref_SBP": baseline_ref_SBP,
    "baseline_ref_source": baseline_ref_source,
    "baseline_protocol_SBP_supine": baseline_protocol,
    "baseline_stable_clean_resting": baseline_stable,
    "baseline_low_motion_clean_resting": baseline_low_motion,
    "baseline_global_resting": baseline_global,
    "resting_analyzable_seconds": int(resting_analyzable.sum()),
    "stable_clean_seconds": int(stable_clean_resting.sum()),
    "low_motion_clean_seconds": int(low_motion_clean_resting.sum()),
    "BASELINE_ACTIVE_METHOD": active,
}])

print("SECTION 5.1 loaded.")
display(baseline_candidates)
display(baseline_model_tbl)

In [ ]:
# =========================
# SECTION 5.2 — BASELINE SENSITIVITY
# Evaluate how alternative baseline definitions affect baseline-relative
# rise and drop metrics, episode counts, and intermediate-quality-fraction summaries.
# =========================

qc = master["QC_status"].astype("object")

sbp = pd.to_numeric(master["SBP"], errors="coerce")
an_thr = master.get("analyzable_for_threshold", master["analyzable"].fillna(False)).fillna(False).astype(bool) & sbp.notna()

eff = master.get("effort_context_mask", pd.Series(False, index=master.index)).fillna(False).astype(bool)
post_win = master.get("posture_window_after_upright", pd.Series(False, index=master.index)).fillna(False).astype(bool)

def intermediate_quality_fraction(mask: pd.Series) -> float:
    """
    Compute the fraction of seconds within a mask that are classified as Intermediate-Quality.
    """
    mask = pd.Series(mask, index=master.index).fillna(False).astype(bool)
    denom = int(mask.sum())
    if denom == 0:
        return np.nan
    return float(qc.loc[mask].eq("Intermediate-Quality").mean())


def count_episodes(mask: pd.Series, min_sec: int) -> int:
    """
    Count bridged contiguous episodes that meet a minimum duration criterion.
    """
    s = bridge_short_false_gaps(mask, CFG["BRIDGE_GAP_SEC"])
    if not s.any():
        return 0

    gid = s.ne(s.shift()).cumsum()
    n = 0
    for _, block in s.groupby(gid):
        if not bool(block.iloc[0]):
            continue
        dur = int((block.index[-1] - block.index[0]).total_seconds()) + 1
        if dur >= min_sec:
            n += 1
    return n


def eval_baseline(b: float) -> dict:
    """
    Evaluate baseline-relative rise and drop metrics for a given baseline SBP.
    """
    if pd.isna(b):
        return {}

    rise = sbp - float(b)
    drop = float(b) - sbp

    BP_RISE20 = an_thr & (rise >= CFG["AD_RISE"])
    BP_RISE30 = an_thr & (rise >= CFG["AD_RISE_HIGH"])
    BP_DROP20 = an_thr & (drop >= CFG["OH_DROP"])

    AD_resting = BP_RISE20 & (~eff)
    EFFORT_PRESSOR = BP_RISE20 & (eff)

    # These are baseline-relative drops and are not equivalent to
    # posture-anchored OH classification unless they occur within the
    # posture window.
    DROP_posture_window = BP_DROP20 & post_win
    DROP_nonanchored = BP_DROP20 & (~post_win)

    return {
        "BP_RISE20_seconds": int(BP_RISE20.sum()),
        "BP_RISE30_seconds": int(BP_RISE30.sum()),
        "BP_DROP20_vs_baseline_seconds": int(BP_DROP20.sum()),

        "AD_resting_seconds": int(AD_resting.sum()),
        "EFFORT_PRESSOR_seconds": int(EFFORT_PRESSOR.sum()),

        "DROP_posture_window_seconds": int(DROP_posture_window.sum()),
        "DROP_nonanchored_seconds": int(DROP_nonanchored.sum()),
        "drop20_prop_in_posture_window": float(DROP_posture_window.sum()) / max(1, int(BP_DROP20.sum())),

        "AD_resting_primary_episodes": count_episodes(AD_resting, CFG["PRIMARY_MIN_SEC"]),
        "EFFORT_PRESSOR_primary_episodes": count_episodes(EFFORT_PRESSOR, CFG["PRIMARY_MIN_SEC"]),
        "DROP_posture_window_primary_episodes": count_episodes(DROP_posture_window, CFG["PRIMARY_MIN_SEC"]),
        "DROP_nonanchored_primary_episodes": count_episodes(DROP_nonanchored, CFG["PRIMARY_MIN_SEC"]),

        "rise20_prop_inside_effort": float((BP_RISE20 & eff).sum()) / max(1, int(BP_RISE20.sum())),
        "rise20_intermediate_quality_fraction": intermediate_quality_fraction(BP_RISE20),
        "ad_resting_intermediate_quality_fraction": intermediate_quality_fraction(AD_resting),
        "effort_pressor_intermediate_quality_fraction": intermediate_quality_fraction(EFFORT_PRESSOR),
    }


rows = []
for _, r in baseline_candidates.iterrows():
    src = str(r["source"])
    b = r["baseline_SBP"]
    out = eval_baseline(b)
    if not out:
        continue
    out.update({
        "patient_id": patient_id,
        "baseline_source": src,
        "baseline_SBP": float(b),
        "baseline_ref_source": baseline_ref_source,
        "delta_vs_baseline_ref": float(b) - float(baseline_ref_SBP) if pd.notna(baseline_ref_SBP) else np.nan,
    })
    rows.append(out)

baseline_sensitivity_tbl = pd.DataFrame(rows).sort_values("baseline_source").reset_index(drop=True)

print("SECTION 5.2 loaded.")
display(baseline_sensitivity_tbl)

In [ ]:
# =========================
# SECTION 5.3 — DIURNAL DESCRIPTORS
# Context-based summary of wake and sleep SBP distributions.
# These outputs are descriptive and exploratory only.
# =========================

def median_masked(series: pd.Series, mask: pd.Series) -> float:
    """
    Return the median of a series within a specified boolean mask.
    """
    s = pd.to_numeric(series[mask], errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan

sleep_mask = master["is_sleep_ctx"].fillna(False)
wake_mask = master["is_wake_ctx"].fillna(False)
reliable = bool(master["sleepwake_context_reliable"].iloc[0])

base_mask = master["analyzable"].fillna(False) & sbp.notna()
SBP_wake_med = median_masked(master["SBP"], base_mask & wake_mask)
SBP_sleep_med = median_masked(master["SBP"], base_mask & sleep_mask)

dipping_pct = np.nan
if reliable and pd.notna(SBP_wake_med) and pd.notna(SBP_sleep_med) and SBP_wake_med != 0:
    dipping_pct = ((SBP_wake_med - SBP_sleep_med) / SBP_wake_med) * 100

diurnal_tbl = pd.DataFrame([{
    "patient_id": patient_id,
    "sleepwake_context_reliable": reliable,
    "SBP_wake_med": SBP_wake_med,
    "SBP_sleep_med": SBP_sleep_med,
    "dipping_pct_exploratory": dipping_pct,
}])

print("SECTION 5.3 loaded.")
display(diurnal_tbl)

In [ ]:
# =========================
# SECTION 6 — CORE FLAGS
# Derive core baseline-relative rise and drop flags, effort-associated pressor flags, posture-anchored drop flags, and descriptive high/low blood-pressure states.
# =========================

if pd.isna(baseline_ref_SBP):
    raise RuntimeError("baseline_ref_SBP is NaN. Check baseline candidates/protocol extraction.")

sbp = pd.to_numeric(master["SBP"], errors="coerce")
dbp = pd.to_numeric(master["DBP"], errors="coerce")
mapv = pd.to_numeric(master["MAP"], errors="coerce") if "MAP" in master.columns else pd.Series(np.nan, index=master.index)

# Prefer analyzable_for_threshold when available so that threshold-based
# flags respect the L4 trigger guard.
an_thr = master.get("analyzable_for_threshold", master.get("analyzable", pd.Series(False, index=master.index)))
an_thr = pd.Series(an_thr, index=master.index).fillna(False).astype(bool) & sbp.notna()

# -------------------------
# Helper functions
# -------------------------
def _bridge_short_false_gaps(mask: pd.Series, max_gap_sec: int) -> pd.Series:
    """
    Fill short False gaps within a boolean mask to preserve continuity
    across brief interruptions.
    """
    s = pd.Series(mask, index=master.index).fillna(False).astype(bool).copy()
    if s.sum() == 0 or max_gap_sec <= 0:
        return s
    grp = s.ne(s.shift()).cumsum()
    blocks = []
    for _, block in s.groupby(grp):
        blocks.append({"val": bool(block.iloc[0]), "start": block.index[0], "end": block.index[-1], "len": len(block)})
    for i in range(1, len(blocks) - 1):
        prev_b, cur_b, next_b = blocks[i - 1], blocks[i], blocks[i + 1]
        if (not cur_b["val"]) and prev_b["val"] and next_b["val"] and cur_b["len"] <= max_gap_sec:
            s.loc[cur_b["start"]:cur_b["end"]] = True
    return s


def _extract_runs(mask: pd.Series, min_len_sec: int) -> pd.DataFrame:
    """
    Extract contiguous True runs from a boolean mask and retain only
    those meeting the minimum duration criterion.
    """
    s = pd.Series(mask, index=master.index).fillna(False).astype(bool)
    if not s.any():
        return pd.DataFrame(columns=["start", "end", "duration_sec"])
    gid = s.ne(s.shift()).cumsum()
    rows = []
    for _, block in s.groupby(gid):
        if not bool(block.iloc[0]):
            continue
        start, end = block.index[0], block.index[-1]
        dur = int((end - start).total_seconds()) + 1
        if dur >= min_len_sec:
            rows.append({"start": start, "end": end, "duration_sec": dur})
    return pd.DataFrame(rows)


def _min_duration_mask(mask: pd.Series, min_sec: int) -> pd.Series:
    """
    Retain only contiguous True segments that satisfy the minimum
    duration requirement.
    """
    s = pd.Series(mask, index=master.index).fillna(False).astype(bool)
    if not s.any():
        return pd.Series(False, index=master.index)
    out = pd.Series(False, index=master.index)
    gid = s.ne(s.shift()).cumsum()
    for _, block in s.groupby(gid):
        if not bool(block.iloc[0]):
            continue
        dur = int((block.index[-1] - block.index[0]).total_seconds()) + 1
        if dur >= min_sec:
            out.loc[block.index[0]:block.index[-1]] = True
    return out

# -------------------------
# Absolute rise relative to reference baseline
# -------------------------
master["SBP_rise_abs"] = sbp - float(baseline_ref_SBP)
master["SBP_drop_abs"] = float(baseline_ref_SBP) - sbp

master["BP_RISE20_abs"] = an_thr & (master["SBP_rise_abs"] >= CFG["AD_RISE"])
master["BP_RISE30_abs"] = an_thr & (master["SBP_rise_abs"] >= CFG["AD_RISE_HIGH"])

# -------------------------
# Local drop relative to the rolling baseline
# Reported descriptively and not treated as OH diagnosis
# -------------------------
sbp_local_base = pd.to_numeric(master["SBP_med120"], errors="coerce")
master["SBP_drop_local"] = sbp_local_base - sbp
master["BP_DROP20_local"] = an_thr & sbp_local_base.notna() & (master["SBP_drop_local"] >= CFG["OH_DROP"])

# -------------------------
# Any-motion sustained bouts
# Descriptive context only; not used to exclude AD_resting
# -------------------------
ANY_MOTION_MIN_SEC = 15
ANY_MOTION_GAP_BRIDGE_SEC = 3
ANY_MOTION_RECOVERY_SEC = 10

if "abs_low_motion" not in master.columns:
    raise RuntimeError("abs_low_motion is missing. Run Section 3 first.")

any_motion_raw = (~master["abs_low_motion"].fillna(False).astype(bool))
any_motion_raw = _bridge_short_false_gaps(any_motion_raw, ANY_MOTION_GAP_BRIDGE_SEC)
any_motion_runs = _extract_runs(any_motion_raw, ANY_MOTION_MIN_SEC)

any_motion_bout_mask = pd.Series(False, index=master.index)
for _, b in any_motion_runs.iterrows():
    s0 = pd.to_datetime(b["start"])
    e0 = pd.to_datetime(b["end"]) + pd.Timedelta(seconds=ANY_MOTION_RECOVERY_SEC)
    any_motion_bout_mask.loc[s0:e0] = True

master["any_motion_bout_mask"] = any_motion_bout_mask.astype(bool)

buf = int(CFG.get("ONSET_BUF_SEC", 10))
master["any_motion_buf10"] = (
    master["any_motion_bout_mask"]
    .rolling(2 * buf + 1, center=True, min_periods=1)
    .max().fillna(0).astype(bool)
)

# -------------------------
# Motion and effort context inputs
# Broad motion context is descriptive only.
# Effort-likely context is used to route rises away from AD_resting.
# -------------------------
motion_ctx = master.get(
    "motion_context_mask",
    master.get("effort_context_mask", pd.Series(False, index=master.index))
)
motion_ctx = pd.Series(motion_ctx, index=master.index).fillna(False).astype(bool)

eff_likely = master.get(
    "effort_likely_mask",
    master.get("effort_context_mask", pd.Series(False, index=master.index))
)
eff_likely = pd.Series(eff_likely, index=master.index).fillna(False).astype(bool)

# -------------------------
# Resting diagnostic eligibility
# Excludes effort-likely periods and clear signal instability.
# Broad motion context is not excluded at this stage.
# -------------------------
ptt_noisy_buf = master.get("ptt_noisy_buf10", pd.Series(False, index=master.index)).fillna(False).astype(bool)

motion_noisy_buf = (
    master.get("high_motion_buf10", pd.Series(False, index=master.index)).fillna(False).astype(bool)
    | master.get("very_high_motion_buf10", pd.Series(False, index=master.index)).fillna(False).astype(bool)
)

master["resting_diag_eligible"] = (
    an_thr
    & (~eff_likely)
    & (~ptt_noisy_buf)
    & (~motion_noisy_buf)
)

# -------------------------
# Physiology support mask
# Used to reduce leakage of unsupported AD_resting classifications
# -------------------------
REF_SEC = 120
map_ref = mapv.rolling(REF_SEC, min_periods=max(30, REF_SEC // 2)).median().shift(1)
dbp_ref = dbp.rolling(REF_SEC, min_periods=max(30, REF_SEC // 2)).median().shift(1)

MAP_RISE_THR = CFG.get("AD_ONSET_MAP_RISE", 5)
DBP_RISE_THR = CFG.get("AD_ONSET_DBP_RISE", 3)

map_support = mapv.notna() & map_ref.notna() & ((mapv - map_ref) >= MAP_RISE_THR)
dbp_support = dbp.notna() & dbp_ref.notna() & ((dbp - dbp_ref) >= DBP_RISE_THR)

ptt_sd = pd.to_numeric(master.get("PTTraw_sd", np.nan), errors="coerce")
ptt_sd_med = float(ptt_sd.dropna().median()) if ptt_sd.notna().any() else np.nan
ptt_sd_mad = float((ptt_sd.dropna() - ptt_sd_med).abs().median()) if ptt_sd.notna().any() else np.nan

PTT_STABLE_THR = np.nan if pd.isna(ptt_sd_med) else (ptt_sd_med + 3.0 * (0 if pd.isna(ptt_sd_mad) else ptt_sd_mad))
ptt_stable = ptt_sd.notna() & (ptt_sd <= PTT_STABLE_THR)

phys_support = (map_support | dbp_support) & ptt_stable
phys_support = _bridge_short_false_gaps(phys_support, max_gap_sec=2)

master["AD_phys_support_sec"] = phys_support.astype(bool)
master["AD_phys_support_buf10"] = (
    master["AD_phys_support_sec"]
    .rolling(2 * buf + 1, center=True, min_periods=1)
    .max().fillna(0).astype(bool)
)

# -------------------------
# Core classification flags
# -------------------------
master["AD_resting_core"] = master["BP_RISE20_abs"] & master["resting_diag_eligible"] & master["AD_phys_support_buf10"]
master["EFFORT_PRESSOR_core"] = master["BP_RISE20_abs"] & eff_likely

# Retain descriptive motion and effort context for downstream reporting.
master["motion_context_mask"] = motion_ctx.astype(bool)
master["effort_likely_mask"] = eff_likely.astype(bool)

# -------------------------
# Posture-anchored drops
# Drops are referenced to a pre-upright baseline and restricted to anchored posture windows.
# -------------------------
PRE_SEC = 60
post_win = master.get("posture_window_after_upright", pd.Series(False, index=master.index)).fillna(False).astype(bool)
pre_ref = sbp.rolling(PRE_SEC, min_periods=max(10, PRE_SEC // 3)).median().shift(1)

master["SBP_pre_upright_ref"] = pre_ref
master["SBP_drop_posture"] = master["SBP_pre_upright_ref"] - sbp
master["BP_DROP20_posture"] = an_thr & post_win & master["SBP_pre_upright_ref"].notna() & (master["SBP_drop_posture"] >= CFG["OH_DROP"])
master["OH_posture_anchored_core"] = master["BP_DROP20_posture"]

# Local non-anchored drops remain descriptive and are not equivalent to OH diagnosis.
master["BP_drop_local_nonanchored_core"] = master["BP_DROP20_local"] & (~post_win)

# -------------------------
# Descriptive high and low blood-pressure ranges and states
# Range flags use analyzable seconds.
# State flags require valid-only segments sustained for the configured duration.
# -------------------------
master["HighBP_range_sec"] = an_thr & ((sbp >= CFG["HTN_SBP"]) | (dbp >= CFG["HTN_DBP"]))
master["LowBP_range_sec"]  = an_thr & ((sbp <= CFG["HYPOTENSION_SBP"]) | (dbp <= CFG["HYPOTENSION_DBP"]))

qc = master.get("QC_status", pd.Series("valid", index=master.index)).astype("object")
valid_only = qc.eq("valid")

MIN_BP_STATE_SEC = int(CFG.get("BP_STATE_MIN_SEC", 60))
GAP_BRIDGE_SEC_STATE = 2

high_state_raw = _bridge_short_false_gaps(master["HighBP_range_sec"] & valid_only, GAP_BRIDGE_SEC_STATE)
low_state_raw  = _bridge_short_false_gaps(master["LowBP_range_sec"] & valid_only, GAP_BRIDGE_SEC_STATE)

master["HighBP_state_core"] = _min_duration_mask(high_state_raw, MIN_BP_STATE_SEC)
master["LowBP_state_core"]  = _min_duration_mask(low_state_raw, MIN_BP_STATE_SEC)

# Supine descriptive and state-specific high-BP outputs.
master["Supine_highBP_range_sec"] = master["HighBP_range_sec"] & master.get("pos_supine_reliable", pd.Series(False, index=master.index)).fillna(False).astype(bool)
master["Supine_highBP_state_core"] = master["HighBP_state_core"] & master.get("pos_supine_reliable", pd.Series(False, index=master.index)).fillna(False).astype(bool)

print("SECTION 6 loaded.")
print("Any-motion runs:", len(any_motion_runs), "| any_motion_bout_mask seconds:", int(master["any_motion_bout_mask"].sum()))
print("PTT stable thr used:", PTT_STABLE_THR)

print("motion_context seconds:", int(master["motion_context_mask"].sum()))
print("effort_likely seconds:", int(master["effort_likely_mask"].sum()))

print("AD_resting seconds:", int(master["AD_resting_core"].sum()))
print("EFFORT_PRESSOR seconds:", int(master["EFFORT_PRESSOR_core"].sum()))
print("OH_posture_anchored seconds:", int(master["OH_posture_anchored_core"].sum()))
print("BP_drop_local_nonanchored seconds:", int(master["BP_drop_local_nonanchored_core"].sum()))
print("HighBP_range seconds:", int(master["HighBP_range_sec"].sum()))
print(f"HighBP_state seconds (valid + >= {MIN_BP_STATE_SEC}s):", int(master["HighBP_state_core"].sum()))
print("LowBP_range seconds:", int(master["LowBP_range_sec"].sum()))
print(f"LowBP_state seconds (valid + >= {MIN_BP_STATE_SEC}s):", int(master["LowBP_state_core"].sum()))

In [ ]:
# =========================
# SECTION 6.1 — QA CHECKS
# Summarise overlap between AD_resting classifications and key exclusion or instability-related context flags.
# =========================

RUN_QA = True

if RUN_QA:
    ad = master.get("AD_resting_core", pd.Series(False, index=master.index)).fillna(False).astype(bool)
    eff = master.get("effort_context_mask", pd.Series(False, index=master.index)).fillna(False).astype(bool)
    ptt = master.get("ptt_noisy_buf10", pd.Series(False, index=master.index)).fillna(False).astype(bool)
    hm  = master.get("high_motion_buf10", pd.Series(False, index=master.index)).fillna(False).astype(bool)
    vhm = master.get("very_high_motion_buf10", pd.Series(False, index=master.index)).fillna(False).astype(bool)

    print("QA — AD_resting seconds:", int(ad.sum()))
    print("QA — overlaps effort_context_mask:", int((ad & eff).sum()))
    print("QA — overlaps ptt_noisy_buf10:", int((ad & ptt).sum()))
    print("QA — overlaps high_motion_buf10:", int((ad & hm).sum()))
    print("QA — overlaps very_high_motion_buf10:", int((ad & vhm).sum()))

    act = pd.to_numeric(master.get("Activity", pd.Series(np.nan, index=master.index)), errors="coerce")
    print("QA — Median Activity during AD:", float(act[ad].median()) if ad.any() else np.nan)
    print("QA — 95th pct Activity during AD:", float(act[ad].quantile(0.95)) if ad.any() else np.nan)

In [ ]:
# =========================
# SECTION 7 — EPISODE EXTRACTION
# Convert second-level event flags into contiguous episodes and summarise
# episode duration, QC composition, and peak haemodynamic features.
# =========================

def bridge_short_false_gaps(mask: pd.Series, max_gap_sec: int) -> pd.Series:
    """
    Fill short False gaps within a boolean mask to preserve continuity
    across brief interruptions.
    """
    s = pd.Series(mask, index=master.index).fillna(False).astype(bool).copy()
    if s.sum() == 0 or max_gap_sec <= 0:
        return s

    grp = s.ne(s.shift()).cumsum()
    blocks = []
    for _, block in s.groupby(grp):
        blocks.append({
            "val": bool(block.iloc[0]),
            "start": block.index[0],
            "end": block.index[-1],
            "len": len(block)
        })

    for i in range(1, len(blocks) - 1):
        prev_b, cur_b, next_b = blocks[i - 1], blocks[i], blocks[i + 1]
        if (not cur_b["val"]) and prev_b["val"] and next_b["val"] and cur_b["len"] <= max_gap_sec:
            s.loc[cur_b["start"]:cur_b["end"]] = True
    return s


def extract_episodes(flag: pd.Series, label: str, min_duration_sec: int = 1) -> pd.DataFrame:
    """
    Extract bridged contiguous episodes from a second-level flag and
    summarise duration, QC composition, and peak signal features.
    """
    s = bridge_short_false_gaps(flag, CFG["BRIDGE_GAP_SEC"])
    if not s.any():
        return pd.DataFrame(columns=[
            "label", "start", "end", "duration_sec",
            "pct_valid", "pct_intermediate_quality", "pct_artefact",
            "peak_sbp", "peak_dbp", "peak_map",
            "peak_sbp_rise_abs",
            "peak_sbp_drop_abs",
            "peak_sbp_drop_local",
            "peak_sbp_drop_posture",
        ])

    gid = s.ne(s.shift()).cumsum()
    rows = []
    for _, block in s.groupby(gid):
        if not bool(block.iloc[0]):
            continue

        start, end = block.index[0], block.index[-1]
        dur = int((end - start).total_seconds()) + 1
        if dur < min_duration_sec:
            continue

        seg = master.loc[start:end]

        # QC composition within the episode window.
        qc_col = seg["QC_status"] if "QC_status" in seg.columns else pd.Series("valid", index=seg.index)
        pct_valid = float((qc_col == "valid").mean() * 100)
        pct_intermediate_quality = float((qc_col == "Intermediate-Quality").mean() * 100)
        pct_art = float((qc_col == "artefact").mean() * 100)

        # Peak signal values within the episode window.
        peak_sbp = float(pd.to_numeric(seg.get("SBP", np.nan), errors="coerce").max())
        peak_dbp = float(pd.to_numeric(seg.get("DBP", np.nan), errors="coerce").max())
        peak_map = float(pd.to_numeric(seg.get("MAP", np.nan), errors="coerce").max()) if "MAP" in seg.columns else np.nan

        peak_rise_abs = float(pd.to_numeric(seg.get("SBP_rise_abs", np.nan), errors="coerce").max())
        peak_drop_abs = float(pd.to_numeric(seg.get("SBP_drop_abs", np.nan), errors="coerce").max())

        peak_drop_local = float(pd.to_numeric(seg.get("SBP_drop_local", np.nan), errors="coerce").max()) if "SBP_drop_local" in seg.columns else np.nan
        peak_drop_posture = float(pd.to_numeric(seg.get("SBP_drop_posture", np.nan), errors="coerce").max()) if "SBP_drop_posture" in seg.columns else np.nan

        rows.append({
            "label": label,
            "start": start,
            "end": end,
            "duration_sec": dur,

            "pct_valid": pct_valid,
            "pct_intermediate_quality": pct_intermediate_quality,
            "pct_artefact": pct_art,

            "peak_sbp": peak_sbp,
            "peak_dbp": peak_dbp,
            "peak_map": peak_map,

            "peak_sbp_rise_abs": peak_rise_abs,
            "peak_sbp_drop_abs": peak_drop_abs,
            "peak_sbp_drop_local": peak_drop_local,
            "peak_sbp_drop_posture": peak_drop_posture,
        })

    return pd.DataFrame(rows)


# Registry of event flags to be converted into episode-level tables.
EVENT_REGISTRY = {
    # Pressor events
    "AD_resting": master["AD_resting_core"].fillna(False),
    "EFFORT_PRESSOR": master["EFFORT_PRESSOR_core"].fillna(False),

    # Drop events
    "OH_posture_anchored": master["OH_posture_anchored_core"].fillna(False),
    "BP_drop_local_nonanchored": master["BP_drop_local_nonanchored_core"].fillna(False),

    # Descriptive blood-pressure range and state summaries
    "HighBP_range": master["HighBP_range_sec"].fillna(False),
    "HighBP_state": master["HighBP_state_core"].fillna(False),
    "LowBP_range": master["LowBP_range_sec"].fillna(False),
    "LowBP_state": master["LowBP_state_core"].fillna(False),

    # Supine-specific descriptive outputs
    "Supine_highBP_range": master["Supine_highBP_range_sec"].fillna(False),
    "Supine_highBP_state": master["Supine_highBP_state_core"].fillna(False),
}

episode_tables = {k: extract_episodes(v, k, 1) for k, v in EVENT_REGISTRY.items()}

print("SECTION 7 loaded. Episode counts:")
for k, df in episode_tables.items():
    print(" -", k, len(df))

In [ ]:
# =========================
# SECTION 8 — ADJUDICATION
# Apply episode-level adjudication using onset features, QC composition,
# physiological support, and event-specific confidence scoring.
# =========================

EXTRA_PEN = int(CFG.get("EXTRA_INTERMEDIATE_QUALITY_PTT_ONSET_PENALTY", 1))

# Set to a subset of event types to adjudicate only selected classes.
# Use None to adjudicate all available event types.
ADJUDICATE_ONLY = None
# Example:
# ADJUDICATE_ONLY = {"AD_resting", "EFFORT_PRESSOR", "OH_posture_anchored", "BP_drop_local_nonanchored"}

def safe_bool(x) -> pd.Series:
    """
    Coerce input to a boolean series with missing values treated as False.
    """
    return pd.Series(x).fillna(False).astype(bool)


def num_max(x) -> float:
    """
    Return the numeric maximum of a series, or NaN if no valid values exist.
    """
    s = pd.to_numeric(x, errors="coerce").dropna()
    return float(s.max()) if len(s) else np.nan


def num_min(x) -> float:
    """
    Return the numeric minimum of a series, or NaN if no valid values exist.
    """
    s = pd.to_numeric(x, errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan


def pre_event_ref(start: pd.Timestamp, col: str, lookback_sec: int = 120) -> float:
    """
    Compute a pre-event reference value as the median over the lookback
    interval preceding the event start.
    """
    if col not in master.columns:
        return np.nan
    seg = master.loc[:start, col].tail(int(lookback_sec))
    s = pd.to_numeric(seg, errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan


def get_ptt_stable_thr() -> float:
    """
    Return the PTT stability threshold.

    Preference order:
    1. PTT_STABLE_THR already defined in the global workspace
    2. Fallback threshold computed as median + 3*MAD of PTTraw_sd
    """
    if "PTT_STABLE_THR" in globals() and pd.notna(globals()["PTT_STABLE_THR"]):
        return float(globals()["PTT_STABLE_THR"])

    if "PTTraw_sd" not in master.columns:
        return np.nan

    ptt_sd = pd.to_numeric(master["PTTraw_sd"], errors="coerce").dropna()
    if len(ptt_sd) == 0:
        return np.nan

    med = float(ptt_sd.median())
    mad = float((ptt_sd - med).abs().median())
    return float(med + 3.0 * mad)


PTT_STABLE_THR_LOCAL = get_ptt_stable_thr()

# Onset physiology thresholds from the analysis configuration.
AD_ONSET_WINDOW_SEC = int(CFG.get("AD_ONSET_WINDOW_SEC", 20))
AD_ONSET_MAP_RISE = float(CFG.get("AD_ONSET_MAP_RISE", 5))
AD_ONSET_DBP_RISE = float(CFG.get("AD_ONSET_DBP_RISE", 3))
AD_MIN_CONCORDANCE_FRAC = float(CFG.get("AD_MIN_CONCORDANCE_FRAC", 0.60))
AD_MIN_PTT_STABLE_FRAC = float(CFG.get("AD_MIN_PTT_STABLE_FRAC", 0.80))

PRIMARY_MIN_SEC = int(CFG.get("PRIMARY_MIN_SEC", 5))


def adjudicate(df: pd.DataFrame, event_type: str) -> pd.DataFrame:
    """
    Adjudicate extracted episodes for a given event type using
    onset features, physiological support, and QC-based scoring.
    """
    if df is None or len(df) == 0:
        return pd.DataFrame()

    out_rows = []

    for _, ev in df.iterrows():
        s = pd.to_datetime(ev["start"])
        e = pd.to_datetime(ev["end"])
        seg = master.loc[s:e]
        if len(seg) == 0:
            continue

        dur = int(pd.to_numeric(ev.get("duration_sec", 0), errors="coerce") or 0)

        # Onset windows
        onset_end9 = min(e, s + pd.Timedelta(seconds=9))
        onset9 = master.loc[s:onset_end9]

        onset_endN = min(e, s + pd.Timedelta(seconds=AD_ONSET_WINDOW_SEC - 1))
        onsetN = master.loc[s:onset_endN]

        # Onset noise and motion flags
        ptt_noisy_onset = bool(safe_bool(onset9.get("ptt_noisy_buf10", False)).any())
        motion_noisy_onset = bool(safe_bool(onset9.get("high_motion_buf10", False)).any()) or bool(
            safe_bool(onset9.get("very_high_motion_buf10", False)).any()
        )

        dominant_intermediate_quality = (
            bool(seg["QC_status"].eq("Intermediate-Quality").mean() > 0.5)
            if "QC_status" in seg.columns else False
        )

        spike_like = bool(
            safe_bool(pd.to_numeric(seg.get("SBP_diff_1s", np.nan), errors="coerce").abs() > CFG["SBP_SPIKE_HARD"]).any()
        )

        # MAP/DBP concordance across the full event
        map_ref = pre_event_ref(s, "MAP", 120)
        dbp_ref = pre_event_ref(s, "DBP", 120)

        map_dbp_concordant = False
        if event_type in ["AD_resting", "EFFORT_PRESSOR", "HighBP_range", "HighBP_state", "Supine_highBP_range", "Supine_highBP_state"]:
            map_up = ((pd.to_numeric(seg.get("MAP", np.nan), errors="coerce") - map_ref) >= 5).mean() >= 0.5 if pd.notna(map_ref) and "MAP" in seg.columns else False
            dbp_up = ((pd.to_numeric(seg.get("DBP", np.nan), errors="coerce") - dbp_ref) >= 3).mean() >= 0.5 if pd.notna(dbp_ref) else False
            map_dbp_concordant = bool(map_up or dbp_up)

        elif event_type in ["OH_posture_anchored", "BP_drop_local_nonanchored", "LowBP_range", "LowBP_state"]:
            map_down = ((map_ref - pd.to_numeric(seg.get("MAP", np.nan), errors="coerce")) >= 5).mean() >= 0.5 if pd.notna(map_ref) and "MAP" in seg.columns else False
            dbp_down = ((dbp_ref - pd.to_numeric(seg.get("DBP", np.nan), errors="coerce")) >= 3).mean() >= 0.5 if pd.notna(dbp_ref) else False
            map_dbp_concordant = bool(map_down or dbp_down)

        # HR support
        min_hr_change = num_min(seg.get("HR_change_10s", pd.Series(dtype=float)))
        max_hr_change = num_max(seg.get("HR_change_10s", pd.Series(dtype=float)))

        hr_support = pd.NA
        if event_type in ["AD_resting", "EFFORT_PRESSOR"]:
            hr_support = (pd.notna(min_hr_change) and min_hr_change <= 0)
        elif event_type in ["OH_posture_anchored", "BP_drop_local_nonanchored", "LowBP_range", "LowBP_state"]:
            hr_support = (pd.notna(max_hr_change) and max_hr_change >= CFG["OH_HR_RISE"])

        # =========================
        # AD_resting hard exclusions
        # =========================
        hard_exclude = False
        hard_exclude_reasons = []
        onset_concordance_frac = np.nan
        onset_ptt_stable_frac = np.nan

        if event_type == "AD_resting":
            # Minimum duration requirement
            if dur < PRIMARY_MIN_SEC:
                hard_exclude = True
                hard_exclude_reasons.append("below_primary_min_duration")

            # Noisy onset excludes AD_resting classification
            if ptt_noisy_onset:
                hard_exclude = True
                hard_exclude_reasons.append("ptt_noisy_onset")
            if motion_noisy_onset:
                hard_exclude = True
                hard_exclude_reasons.append("motion_noisy_onset")

            # Sustained onset concordance and PTT stability
            map_refN = pre_event_ref(s, "MAP", 120)
            dbp_refN = pre_event_ref(s, "DBP", 120)

            if len(onsetN) > 0:
                map_support_sec = pd.Series(False, index=onsetN.index)
                dbp_support_sec = pd.Series(False, index=onsetN.index)

                if pd.notna(map_refN) and "MAP" in onsetN.columns:
                    map_support_sec = (pd.to_numeric(onsetN["MAP"], errors="coerce") - map_refN) >= AD_ONSET_MAP_RISE
                if pd.notna(dbp_refN) and "DBP" in onsetN.columns:
                    dbp_support_sec = (pd.to_numeric(onsetN["DBP"], errors="coerce") - dbp_refN) >= AD_ONSET_DBP_RISE

                concordant = (map_support_sec.fillna(False) | dbp_support_sec.fillna(False))
                onset_concordance_frac = float(concordant.mean()) if len(concordant) else 0.0

                if "PTTraw_sd" in onsetN.columns and pd.notna(PTT_STABLE_THR_LOCAL):
                    ptt_sdN = pd.to_numeric(onsetN["PTTraw_sd"], errors="coerce")
                    ptt_stableN = ptt_sdN.notna() & (ptt_sdN <= PTT_STABLE_THR_LOCAL)
                    onset_ptt_stable_frac = float(ptt_stableN.mean()) if len(ptt_stableN) else 0.0

                if pd.notna(onset_concordance_frac) and onset_concordance_frac < AD_MIN_CONCORDANCE_FRAC:
                    hard_exclude = True
                    hard_exclude_reasons.append("low_onset_concordance")

                if pd.notna(onset_ptt_stable_frac) and onset_ptt_stable_frac < AD_MIN_PTT_STABLE_FRAC:
                    hard_exclude = True
                    hard_exclude_reasons.append("low_onset_ptt_stability")

        # =========================
        # Confidence scoring
        # =========================
        score = 0

        # Magnitude contribution
        if event_type in ["AD_resting", "EFFORT_PRESSOR"]:
            peak = float(ev.get("peak_sbp_rise_abs", np.nan))
            if pd.notna(peak):
                score += 2 if peak >= CFG["AD_RISE_HIGH"] else (1 if peak >= CFG["AD_RISE"] else 0)

        elif event_type in ["OH_posture_anchored", "BP_drop_local_nonanchored"]:
            peak = float(ev.get("peak_sbp_drop_abs", np.nan))
            if pd.notna(peak):
                score += 2 if peak >= 30 else (1 if peak >= 20 else 0)

        # Duration contribution
        score += 2 if dur >= 30 else (1 if dur >= 10 else (1 if dur >= 5 else 0))

        # QC composition contribution
        pct_valid = float(ev.get("pct_valid", np.nan))
        pct_art = float(ev.get("pct_artefact", np.nan))

        if pd.notna(pct_valid):
            score += 2 if pct_valid >= 80 else (1 if pct_valid >= 50 else 0)

        if pd.notna(pct_art) and pct_art > 0:
            score -= 2
        elif dominant_intermediate_quality:
            score -= 1

        # Conditional penalty
        if dominant_intermediate_quality and ptt_noisy_onset:
            score -= EXTRA_PEN

        # Onset support contributions
        score += (1 if not ptt_noisy_onset else -1)
        score += (1 if not motion_noisy_onset else -1)

        if hr_support is True:
            score += 1
        if map_dbp_concordant:
            score += 1
        if spike_like:
            score -= 2

        # Confidence class assignment
        if pd.notna(pct_art) and pct_art > 25:
            conf = "likely_artefact"
        elif score >= 6:
            conf = "high_confidence"
        elif score >= 3:
            conf = "moderate_confidence"
        elif score >= 1:
            conf = "low_confidence"
        else:
            conf = "likely_artefact"

        # Hard override for AD_resting exclusions
        if event_type == "AD_resting" and hard_exclude:
            conf = "context_excluded"

        row = ev.to_dict()
        row.update({
            "support_score": int(score),
            "confidence_class": conf,
            "dominant_intermediate_quality": bool(dominant_intermediate_quality),
            "ptt_noisy_onset": bool(ptt_noisy_onset),
            "motion_noisy_onset": bool(motion_noisy_onset),
            "map_dbp_concordant": bool(map_dbp_concordant),
            "hr_support": hr_support,

            # Onset diagnostics
            "ptt_stable_thr_used": float(PTT_STABLE_THR_LOCAL) if pd.notna(PTT_STABLE_THR_LOCAL) else np.nan,
            "onset_concordance_frac": onset_concordance_frac,
            "onset_ptt_stable_frac": onset_ptt_stable_frac,
            "hard_exclude": bool(hard_exclude) if event_type == "AD_resting" else False,
            "hard_exclude_reasons": ";".join(hard_exclude_reasons) if (event_type == "AD_resting" and hard_exclude_reasons) else "",
        })

        out_rows.append(row)

    return pd.DataFrame(out_rows)


# Run adjudication
if ADJUDICATE_ONLY is None:
    review_tables = {k: adjudicate(v, k) for k, v in episode_tables.items()}
else:
    review_tables = {k: adjudicate(v, k) for k, v in episode_tables.items() if k in ADJUDICATE_ONLY}

print("SECTION 8 loaded. Confidence breakdowns:")
for k, df in review_tables.items():
    if df is None or len(df) == 0:
        print(" -", k, ": no events")
    else:
        print(" -", k)
        display(df["confidence_class"].value_counts(dropna=False).rename_axis("confidence_class").reset_index(name="count"))

if "AD_resting" in review_tables and len(review_tables["AD_resting"]):
    df_ad = review_tables["AD_resting"]
    print("\nAD_resting hard-exclude summary:")
    display(df_ad["hard_exclude"].value_counts(dropna=False).rename_axis("hard_exclude").reset_index(name="count"))
    if (df_ad["hard_exclude"] == True).any():
        print("\nTop hard-exclude reasons:")
        display(
            df_ad.loc[df_ad["hard_exclude"] == True, "hard_exclude_reasons"]
            .value_counts()
            .head(10)
            .rename_axis("reasons")
            .reset_index(name="count")
        )

In [ ]:
# =========================
# SECTION 8.1 — REVIEW PLOTS
# Visual review of selected AD_resting episodes with surrounding contextual signal windows.
# =========================

%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

if "AD_resting" not in review_tables or len(review_tables["AD_resting"]) == 0:
    print("No AD_resting rows in review_tables → nothing to plot.")
else:
    df_ad = review_tables["AD_resting"].copy()
    print("AD_resting confidence breakdown:")
    print(df_ad["confidence_class"].value_counts(dropna=False))

    # Select episodes for plotting.
    df_show = df_ad[df_ad["confidence_class"].isin(["high_confidence", "context_excluded"])].copy()
    print("Events selected for plotting:", len(df_show))

    if len(df_show) == 0:
        print("Filter produced 0 rows. Plotting all AD_resting rows instead.")
        df_show = df_ad.copy()

    def plot_event_window(start, end, pad_sec=180):
        """
        Plot a padded window around an adjudicated event for visual review.
        """
        s0 = pd.to_datetime(start) - pd.Timedelta(seconds=pad_sec)
        e0 = pd.to_datetime(end) + pd.Timedelta(seconds=pad_sec)
        seg = master.loc[s0:e0].copy()

        fig, axes = plt.subplots(5, 1, figsize=(16, 9), sharex=True, constrained_layout=True)

        axes[0].plot(seg.index, seg.get("SBP", np.nan), lw=1)
        axes[0].axvspan(pd.to_datetime(start), pd.to_datetime(end), alpha=0.15, color="dodgerblue")
        axes[0].set_ylabel("SBP")

        if "MAP" in seg.columns:
            axes[1].plot(seg.index, seg["MAP"], lw=1)
            axes[1].axvspan(pd.to_datetime(start), pd.to_datetime(end), alpha=0.15, color="dodgerblue")
            axes[1].set_ylabel("MAP")

        if "DBP" in seg.columns:
            axes[2].plot(seg.index, seg["DBP"], lw=1)
            axes[2].axvspan(pd.to_datetime(start), pd.to_datetime(end), alpha=0.15, color="dodgerblue")
            axes[2].set_ylabel("DBP")

        if "PTTraw" in seg.columns:
            axes[3].plot(seg.index, seg["PTTraw"], lw=1)
            axes[3].axvspan(pd.to_datetime(start), pd.to_datetime(end), alpha=0.15, color="dodgerblue")
            axes[3].set_ylabel("PTTraw")

        if "Activity" in seg.columns:
            axes[4].plot(seg.index, seg["Activity"], lw=1)
            axes[4].axvspan(pd.to_datetime(start), pd.to_datetime(end), alpha=0.15, color="dodgerblue")
            axes[4].set_ylabel("Activity")

        plt.show()

    # Plot the first three selected events.
    for i, (_, r) in enumerate(df_show.iterrows()):
        if i >= 3:
            break
        print(
            "Event:", r["start"], "→", r["end"],
            "| conf:", r.get("confidence_class", ""),
            "| exclude:", r.get("hard_exclude_reasons", "")
        )
        plot_event_window(r["start"], r["end"], pad_sec=180)

In [ ]:
# =========================
# SECTION 9 — FINAL SETS
# Construct final event subsets for primary, sustained, and sensitivity analyses based on adjudication outputs, confidence class, and duration.
# =========================
from typing import Dict

def build_final_event_sets(
    review_df: pd.DataFrame,
    primary_min_sec: int,
    sustained_min_sec: int,
    *,
    exclude_confidence: list = None
) -> Dict[str, pd.DataFrame]:
    """
    Partition adjudicated events into analysis-ready subsets.

    Returns
    -------
    dict
        Dictionary containing:
        - all: all adjudicated events with set membership flags
        - primary: non-excluded events meeting primary criteria
        - sustained: primary events meeting sustained-duration criteria
        - sensitivity: broader non-excluded set including low-confidence events
        - summary: counts and cumulative durations for each set
    """
    if exclude_confidence is None:
        exclude_confidence = ["context_excluded"]

    if review_df is None or len(review_df) == 0:
        empty = pd.DataFrame()
        summary = pd.DataFrame([{
            "raw_total": 0, "primary_total": 0, "sustained_total": 0, "sensitivity_total": 0,
            "excluded_total": 0, "raw_seconds": 0.0, "primary_seconds": 0.0,
            "sustained_seconds": 0.0, "sensitivity_seconds": 0.0
        }])
        return {"all": empty, "primary": empty, "sustained": empty, "sensitivity": empty, "summary": summary}

    df = review_df.copy()
    df["duration_sec"] = pd.to_numeric(df.get("duration_sec", np.nan), errors="coerce")

    if "confidence_class" not in df.columns:
        df["confidence_class"] = "unknown"

    if "hard_exclude" in df.columns:
        df["hard_exclude"] = df["hard_exclude"].fillna(False).astype(bool)
    else:
        df["hard_exclude"] = False

    df["is_excluded"] = df["confidence_class"].isin(exclude_confidence) | df["hard_exclude"]

    primary_conf = ["high_confidence", "moderate_confidence"]
    sensitivity_conf = ["high_confidence", "moderate_confidence", "low_confidence"]

    df["is_primary"] = (
        (~df["is_excluded"])
        & df["confidence_class"].isin(primary_conf)
        & (df["duration_sec"] >= primary_min_sec)
    )

    df["is_sustained"] = df["is_primary"] & (df["duration_sec"] >= sustained_min_sec)

    df["is_sensitivity"] = (
        (~df["is_excluded"])
        & df["confidence_class"].isin(sensitivity_conf)
        & (df["duration_sec"] >= primary_min_sec)
    )

    primary = df[df["is_primary"]].copy()
    sustained = df[df["is_sustained"]].copy()
    sensitivity = df[df["is_sensitivity"]].copy()
    excluded = df[df["is_excluded"]].copy()

    summary = pd.DataFrame([{
        "raw_total": int(len(df)),
        "primary_total": int(len(primary)),
        "sustained_total": int(len(sustained)),
        "sensitivity_total": int(len(sensitivity)),
        "excluded_total": int(len(excluded)),
        "raw_seconds": float(df["duration_sec"].sum(skipna=True)),
        "primary_seconds": float(primary["duration_sec"].sum(skipna=True)),
        "sustained_seconds": float(sustained["duration_sec"].sum(skipna=True)),
        "sensitivity_seconds": float(sensitivity["duration_sec"].sum(skipna=True)),
    }])

    return {"all": df, "primary": primary, "sustained": sustained, "sensitivity": sensitivity, "summary": summary}


final_sets = {}
for name, df0 in review_tables.items():
    sustained_min = CFG["AD_SUSTAINED_MIN_SEC"] if name in ["AD_resting", "EFFORT_PRESSOR"] else CFG["OTHER_SUSTAINED_MIN_SEC"]
    final_sets[name] = build_final_event_sets(df0, CFG["PRIMARY_MIN_SEC"], sustained_min)

print("SECTION 9 loaded. Summaries:")
for name, fs in final_sets.items():
    print(" -", name)
    display(fs["summary"])

if "AD_resting" in final_sets and len(final_sets["AD_resting"]["all"]):
    print("\nAD_resting: confidence_class breakdown (all):")
    display(
        final_sets["AD_resting"]["all"]["confidence_class"]
        .value_counts(dropna=False)
        .rename_axis("confidence_class")
        .reset_index(name="count")
    )
    print("\nAD_resting: excluded reasons breakdown (if present):")
    if "hard_exclude_reasons" in final_sets["AD_resting"]["all"].columns:
        display(
            final_sets["AD_resting"]["all"]
            .loc[final_sets["AD_resting"]["all"]["is_excluded"], "hard_exclude_reasons"]
            .value_counts()
            .rename_axis("reason")
            .reset_index(name="count")
        )

In [ ]:
# =========================
# SECTION 9.1 — BASELINE METHOD SWEEP
# Generate final event summaries for each candidate baseline method.
#
# Requires:
#   - baseline_candidates (Section 5.1)
#   - extract_episodes() (Section 7)
#   - adjudicate() (Section 8)
#   - build_final_event_sets() (Section 9)
#
# Note:
#   This sweep temporarily applies each candidate baseline to the
#   baseline-dependent columns in master, then restores the default
#   baseline at the end.
# =========================

# -------------------------
# Safety checks
# -------------------------
assert "baseline_candidates" in globals(), "Run Section 5.1 first (baseline_candidates missing)."
assert "extract_episodes" in globals(), "Run Section 7 first (extract_episodes missing)."
assert "adjudicate" in globals(), "Run Section 8 first (adjudicate missing)."
assert "build_final_event_sets" in globals(), "Run Section 9 first (build_final_event_sets missing)."

#Storage of per-baseline intermediate tables
STORE_DEBUG = True

def _get_an_thr():
    """
    Return SBP and the threshold-eligible analyzable mask.
    """
    sbp = pd.to_numeric(master["SBP"], errors="coerce")
    an_thr = master.get("analyzable_for_threshold", master.get("analyzable", pd.Series(False, index=master.index)))
    an_thr = pd.Series(an_thr, index=master.index).fillna(False).astype(bool) & sbp.notna()
    return sbp, an_thr


def _apply_baseline_to_master(source: str, bval: float):
    """
    Apply a baseline to master by updating baseline-dependent columns only.

    QC, effort context, resting eligibility, and physiology-support
    fields are left unchanged.
    """
    global baseline_ref_SBP, baseline_ref_source

    baseline_ref_SBP = float(bval)
    baseline_ref_source = str(source)

    master["baseline_ref_SBP"] = baseline_ref_SBP
    master["baseline_ref_source"] = baseline_ref_source

    sbp, an_thr = _get_an_thr()

    master["SBP_rise_abs"] = sbp - float(baseline_ref_SBP)
    master["SBP_drop_abs"] = float(baseline_ref_SBP) - sbp
    master["BP_RISE20_abs"] = an_thr & (master["SBP_rise_abs"] >= CFG["AD_RISE"])
    master["BP_RISE30_abs"] = an_thr & (master["SBP_rise_abs"] >= CFG["AD_RISE_HIGH"])

    # Recompute core labels that depend on absolute baseline-relative rise.
    master["AD_resting_core"] = master["BP_RISE20_abs"] & master["resting_diag_eligible"] & master["AD_phys_support_buf10"]
    master["EFFORT_PRESSOR_core"] = master["BP_RISE20_abs"] & master["effort_context_mask"].fillna(False).astype(bool)


def run_pipeline_for_baseline(source: str, bval: float):
    """
    Run episode extraction, adjudication, and final-set construction
    for a single baseline candidate.

    Returns
    -------
    tuple
        (row_summary, episode_tables_local, review_tables_local, final_sets_local)
    """
    _apply_baseline_to_master(source, bval)

    # Rebuild episode tables for the current baseline.
    EVENT_REGISTRY_LOCAL = {
        "AD_resting": master["AD_resting_core"].fillna(False),
        "EFFORT_PRESSOR": master["EFFORT_PRESSOR_core"].fillna(False),
        "OH_posture_anchored": master.get("OH_posture_anchored_core", pd.Series(False, index=master.index)).fillna(False),
        "BP_drop_local_nonanchored": master.get("BP_drop_local_nonanchored_core", pd.Series(False, index=master.index)).fillna(False),

        "HighBP_range": master.get("HighBP_range_sec", pd.Series(False, index=master.index)).fillna(False),
        "HighBP_state": master.get("HighBP_state_core", pd.Series(False, index=master.index)).fillna(False),
        "LowBP_range": master.get("LowBP_range_sec", pd.Series(False, index=master.index)).fillna(False),
        "LowBP_state": master.get("LowBP_state_core", pd.Series(False, index=master.index)).fillna(False),

        "Supine_highBP_range": master.get("Supine_highBP_range_sec", pd.Series(False, index=master.index)).fillna(False),
        "Supine_highBP_state": master.get("Supine_highBP_state_core", pd.Series(False, index=master.index)).fillna(False),
    }

    episode_tables_local = {k: extract_episodes(v, k, 1) for k, v in EVENT_REGISTRY_LOCAL.items()}

    # Adjudicate episode tables.
    review_tables_local = {k: adjudicate(v, k) for k, v in episode_tables_local.items()}

    # Build final event sets.
    final_sets_local = {}
    for name, df0 in review_tables_local.items():
        sustained_min = CFG["AD_SUSTAINED_MIN_SEC"] if name in ["AD_resting", "EFFORT_PRESSOR"] else CFG["OTHER_SUSTAINED_MIN_SEC"]
        final_sets_local[name] = build_final_event_sets(df0, CFG["PRIMARY_MIN_SEC"], sustained_min)

    def get_summary(event_name: str, key: str):
        try:
            return final_sets_local[event_name]["summary"][key].iloc[0]
        except Exception:
            return np.nan

    row = {
        "patient_id": patient_id,
        "baseline_source": str(source),
        "baseline_SBP": float(bval),

        "AD_resting_primary": get_summary("AD_resting", "primary_total"),
        "AD_resting_sustained": get_summary("AD_resting", "sustained_total"),
        "AD_resting_excluded": get_summary("AD_resting", "excluded_total"),

        "EFFORT_PRESSOR_primary": get_summary("EFFORT_PRESSOR", "primary_total"),
        "EFFORT_PRESSOR_sustained": get_summary("EFFORT_PRESSOR", "sustained_total"),

        "OH_posture_anchored_primary": get_summary("OH_posture_anchored", "primary_total"),
        "BP_drop_local_nonanchored_primary": get_summary("BP_drop_local_nonanchored", "primary_total"),

        "HighBP_state_primary": get_summary("HighBP_state", "primary_total"),
        "LowBP_state_primary": get_summary("LowBP_state", "primary_total"),

        "AD_resting_primary_seconds": get_summary("AD_resting", "primary_seconds"),
        "EFFORT_PRESSOR_primary_seconds": get_summary("EFFORT_PRESSOR", "primary_seconds"),
    }

    return row, episode_tables_local, review_tables_local, final_sets_local


# -------------------------
# Run sweep across baseline candidates
# -------------------------
rows = []
baseline_debug_store = {}

for _, r in baseline_candidates.iterrows():
    src = str(r["source"])
    b = r["baseline_SBP"]
    if pd.isna(b):
        continue

    row, ep_local, rev_local, fs_local = run_pipeline_for_baseline(src, float(b))
    rows.append(row)

    if STORE_DEBUG:
        baseline_debug_store[src] = {
            "episode_tables": ep_local,
            "review_tables": rev_local,
            "final_sets": fs_local
        }

baseline_final_summary_tbl = pd.DataFrame(rows).sort_values("baseline_source").reset_index(drop=True)

print("SECTION 9.1 loaded. Final summaries per baseline method:")
display(baseline_final_summary_tbl)


# -------------------------
# Restore default baseline
# -------------------------
_b0 = baseline_candidates.loc[baseline_candidates["source"].eq("protocol_supine"), "baseline_SBP"]
if len(_b0) == 0 or pd.isna(_b0.iloc[0]):
    raise RuntimeError("protocol_supine baseline not available to restore after sweep.")

_apply_baseline_to_master("protocol_supine", float(_b0.iloc[0]))
print("Baseline restored after sweep:", baseline_ref_source, baseline_ref_SBP)

In [ ]:
# =========================
# SECTION 9.1A — RANGE-TO-STATE QA
# Summarise the proportion of descriptive range seconds that also meet sustained state criteria under the current baseline.
# =========================

print("Baseline now:", master.get("baseline_ref_source", None), master.get("baseline_ref_SBP", None))

high_range = int(master["HighBP_range_sec"].sum())
high_state = int(master["HighBP_state_core"].sum())
low_range  = int(master["LowBP_range_sec"].sum())
low_state  = int(master["LowBP_state_core"].sum())

print("HighBP: % of range seconds that are sustained-state:", 100 * high_state / max(1, high_range))
print("LowBP:  % of range seconds that are sustained-state:", 100 * low_state  / max(1, low_range))

In [ ]:
# =========================
# SECTION 9.2 — EXPORT PACK (PID CONSISTENT + AUTO-SYNC patient_id)
# =========================
import numpy as np
import pandas as pd

def _get(fs, key, default=np.nan):
    try:
        v = fs["summary"][key].iloc[0]
        if isinstance(v, (np.integer, int)): return int(v)
        if isinstance(v, (np.floating, float)): return float(v)
        return v
    except Exception:
        return default

def _fs(name):
    return final_sets.get(name, {"summary": pd.DataFrame([{}])})

def _sanitize_cols(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
        return df
    out = df.copy()
    out.columns = (
        pd.Index(out.columns).astype(str)
        .str.replace("≥", "ge", regex=False)
        .str.replace(">", "gt", regex=False)
        .str.replace("%", "pct", regex=False)
        .str.replace("(", "_", regex=False)
        .str.replace(")", "", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace(" ", "_", regex=False)
    )
    return out

def _pid_from_df(df: pd.DataFrame):
    if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
        return None
    if "patient_id" not in df.columns:
        return None
    vals = [str(x) for x in df["patient_id"].dropna().unique()]
    if len(vals) == 1:
        return vals[0]
    return None

# --- Canonical PID resolution ---
pid_global = str(patient_id)
pid_abpm = _pid_from_df(globals().get("abpm_diagnostic_summary_tbl", None))
PID = pid_abpm if pid_abpm is not None else pid_global

# --- AUTO-SYNC global patient_id to PID (prevents stale IDs flowing into export) ---
if pid_global != PID:
    print(f"AUTO-SYNC: Updating global patient_id from '{pid_global}' -> '{PID}' for this run.")
    patient_id = str(PID)

print("Canonical PID used:", PID)

# ---------- QC summary ----------
qc_tbl = pd.DataFrame([{
    "patient_id": PID,
    "pct_valid": float((master["QC_status"] == "valid").mean() * 100) if "QC_status" in master.columns else np.nan,
    "pct_Intermediate-Quality": float((master["QC_status"] == "Intermediate-Quality").mean() * 100) if "QC_status" in master.columns else np.nan,
    "pct_artefact": float((master["QC_status"] == "artefact").mean() * 100) if "QC_status" in master.columns else np.nan,
    "seconds_total": int(len(master)),
    "seconds_analyzable": int(master.get("analyzable", pd.Series(False, index=master.index)).sum()),
    "seconds_analyzable_for_threshold": int(master.get("analyzable_for_threshold",
                                                      master.get("analyzable", pd.Series(False, index=master.index))).sum()),
    "L4_trigger_guard_seconds": int(master.get("L4_trigger_guard", pd.Series(False, index=master.index)).sum()),
}])

# ---------- Event summary ----------
event_tbl = pd.DataFrame([{
    "patient_id": PID,
    "baseline_ref_SBP": float(master["baseline_ref_SBP"].dropna().iloc[0]) if "baseline_ref_SBP" in master.columns and master["baseline_ref_SBP"].notna().any() else np.nan,
    "baseline_ref_source": str(master["baseline_ref_source"].dropna().iloc[0]) if "baseline_ref_source" in master.columns and master["baseline_ref_source"].notna().any() else None,

    "AD_resting_primary": _get(_fs("AD_resting"), "primary_total", 0),
    "AD_resting_sustained": _get(_fs("AD_resting"), "sustained_total", 0),
    "AD_resting_excluded": _get(_fs("AD_resting"), "excluded_total", 0),

    "EFFORT_PRESSOR_primary": _get(_fs("EFFORT_PRESSOR"), "primary_total", 0),
    "EFFORT_PRESSOR_sustained": _get(_fs("EFFORT_PRESSOR"), "sustained_total", 0),

    "OH_posture_anchored_primary": _get(_fs("OH_posture_anchored"), "primary_total", 0),
    "OH_posture_anchored_sustained": _get(_fs("OH_posture_anchored"), "sustained_total", 0),

    "BP_drop_local_nonanchored_primary": _get(_fs("BP_drop_local_nonanchored"), "primary_total", 0),
    "BP_drop_local_nonanchored_sustained": _get(_fs("BP_drop_local_nonanchored"), "sustained_total", 0),

    "HighBP_range_primary": _get(_fs("HighBP_range"), "primary_total", 0),
    "HighBP_range_sustained": _get(_fs("HighBP_range"), "sustained_total", 0),
    "HighBP_state_primary": _get(_fs("HighBP_state"), "primary_total", 0),
    "HighBP_state_sustained": _get(_fs("HighBP_state"), "sustained_total", 0),

    "LowBP_range_primary": _get(_fs("LowBP_range"), "primary_total", 0),
    "LowBP_range_sustained": _get(_fs("LowBP_range"), "sustained_total", 0),
    "LowBP_state_primary": _get(_fs("LowBP_state"), "primary_total", 0),
    "LowBP_state_sustained": _get(_fs("LowBP_state"), "sustained_total", 0),

    "Supine_highBP_range_primary": _get(_fs("Supine_highBP_range"), "primary_total", 0),
    "Supine_highBP_state_primary": _get(_fs("Supine_highBP_state"), "primary_total", 0),

    "motion_context_seconds": int(master.get("motion_context_mask", pd.Series(False, index=master.index)).sum()),
    "effort_likely_seconds": int(master.get("effort_likely_mask", pd.Series(False, index=master.index)).sum()),
    "any_motion_bout_seconds": int(master.get("any_motion_bout_mask", pd.Series(False, index=master.index)).sum()),
    "posture_window_seconds": int(master.get("posture_window_after_upright", pd.Series(False, index=master.index)).sum()),
}])

# ---------- Run metadata ----------
run_meta_tbl = pd.DataFrame([{
    "patient_id": PID,
    "CFG_VERSION": CFG_VERSION if "CFG_VERSION" in globals() else None,
    "protocol_tol_min": int(globals().get("PROTOCOL_TOL_MIN", 2)),
    "protocol_supine_points_matched": int(protocol_supine_tbl["resolved_time"].notna().sum()) if "protocol_supine_tbl" in globals() and isinstance(protocol_supine_tbl, pd.DataFrame) and "resolved_time" in protocol_supine_tbl.columns else 0,
    "protocol_seated_points_matched": int(protocol_seated_tbl["resolved_time"].notna().sum()) if "protocol_seated_tbl" in globals() and isinstance(protocol_seated_tbl, pd.DataFrame) and "resolved_time" in protocol_seated_tbl.columns else 0,
    "protocol_drift_points_matched": int(protocol_drift_tbl["resolved_time"].notna().sum()) if "protocol_drift_tbl" in globals() and isinstance(protocol_drift_tbl, pd.DataFrame) and "resolved_time" in protocol_drift_tbl.columns else 0,
    "sleepwake_context_reliable": bool(master.get("sleepwake_context_reliable", pd.Series([False])).iloc[0]) if "sleepwake_context_reliable" in master.columns else np.nan,
    "pos_known_pct": float(master.get("Position_ctx", pd.Series(index=master.index)).notna().mean() * 100) if "Position_ctx" in master.columns else np.nan,
    "anchor_thr": float(CFG.get("ANCHOR_THR", 0.60)),
}])

# ---------- Tables ----------
abpm_diagnostic_summary_tbl_out = abpm_diagnostic_summary_tbl.copy() if "abpm_diagnostic_summary_tbl" in globals() else None
if abpm_diagnostic_summary_tbl_out is not None and "patient_id" in abpm_diagnostic_summary_tbl_out.columns:
    abpm_diagnostic_summary_tbl_out = abpm_diagnostic_summary_tbl_out.copy()
    abpm_diagnostic_summary_tbl_out["patient_id"] = PID

posture_feasibility_tbl_out = pos_meta.copy() if "pos_meta" in globals() else None
if posture_feasibility_tbl_out is not None:
    posture_feasibility_tbl_out = posture_feasibility_tbl_out.copy()
    if "patient_id" in posture_feasibility_tbl_out.columns:
        posture_feasibility_tbl_out = posture_feasibility_tbl_out.drop(columns=["patient_id"])
    posture_feasibility_tbl_out.insert(0, "patient_id", PID)

# ---------- Sanitize ----------
qc_tbl = _sanitize_cols(qc_tbl)
event_tbl = _sanitize_cols(event_tbl)
run_meta_tbl = _sanitize_cols(run_meta_tbl)
abpm_diagnostic_summary_tbl_out = _sanitize_cols(abpm_diagnostic_summary_tbl_out) if abpm_diagnostic_summary_tbl_out is not None else None
posture_feasibility_tbl_out = _sanitize_cols(posture_feasibility_tbl_out) if posture_feasibility_tbl_out is not None else None

# ---------- Display ----------
print("qc_tbl"); display(qc_tbl)
print("event_tbl"); display(event_tbl)
print("run_meta_tbl"); display(run_meta_tbl)
if posture_feasibility_tbl_out is not None:
    print("posture_feasibility_tbl"); display(posture_feasibility_tbl_out)
if abpm_diagnostic_summary_tbl_out is not None:
    print("abpm_diagnostic_summary_tbl"); display(abpm_diagnostic_summary_tbl_out)

print("SECTION 9.2 loaded.")

export_pack = {
    "qc_tbl": qc_tbl,
    "event_tbl": event_tbl,
    "run_meta_tbl": run_meta_tbl,
    "posture_feasibility_tbl": posture_feasibility_tbl_out,
    "abpm_diagnostic_summary_tbl": abpm_diagnostic_summary_tbl_out,
}

In [ ]:
# =========================
# SECTION 9.2.1 — DIAGNOSTIC SUMMARY (ABPM-STYLE; INTERPRETATION-READY)
# This section provides an ABPM-style blood-pressure behaviour summary.
# It is descriptive and not intended as AD/OH diagnosis.
# - Uses sleep/wake context when reliable; otherwise applies a fixed-time fallback.
# - ABPM_USE_TRIGGER_GUARD determines whether L4 outlier exclusion is
#   applied to mean and load calculations.
# =========================

import numpy as np
import pandas as pd

# -------------------------
# Configuration
# -------------------------
ABPM_USE_TRIGGER_GUARD = True   # True = use analyzable_for_threshold; False = use analyzable (valid + Intermediate-Quality)
DAY_START = "07:00"             # Fallback wake start if sleep/wake context is unreliable
DAY_END   = "23:00"             # Fallback wake end if sleep/wake context is unreliable

# -------------------------
# Helper functions
# -------------------------
def _safe_mean(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(x.mean()) if len(x) else np.nan

def _safe_median(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(x.median()) if len(x) else np.nan

def _safe_sd(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    return float(x.std(ddof=1)) if len(x) > 1 else np.nan

def _pct(mask):
    mask = pd.Series(mask, index=master.index).fillna(False).astype(bool)
    return float(mask.mean() * 100)

def _secs(mask):
    return int(pd.Series(mask, index=master.index).fillna(False).astype(bool).sum())

def _mins(mask):
    return _secs(mask) / 60.0

# -------------------------
# Base series
# -------------------------
sbp = pd.to_numeric(master.get("SBP", np.nan), errors="coerce")
dbp = pd.to_numeric(master.get("DBP", np.nan), errors="coerce")
mapv = pd.to_numeric(master.get("MAP", np.nan), errors="coerce") if "MAP" in master.columns else pd.Series(np.nan, index=master.index)
hr  = pd.to_numeric(master.get("HR", np.nan), errors="coerce") if "HR" in master.columns else pd.Series(np.nan, index=master.index)

qc = master.get("QC_status", pd.Series("valid", index=master.index)).astype("object")
analyzable = master.get("analyzable", qc.isin(["valid", "Intermediate-Quality"])).fillna(False).astype(bool)
valid_only = qc.eq("valid")

an_thr = master.get("analyzable_for_threshold", analyzable).fillna(False).astype(bool)

# Base mask for ABPM-style summaries, with trigger-guard exclusion.
abpm_base_mask = an_thr if ABPM_USE_TRIGGER_GUARD else analyzable

# -------------------------
# Wake and sleep masks
# -------------------------
reliable = bool(master.get("sleepwake_context_reliable", pd.Series([False])).iloc[0]) if "sleepwake_context_reliable" in master.columns else False
wake_mask = master.get("is_wake_ctx", pd.Series(False, index=master.index)).fillna(False).astype(bool)
sleep_mask = master.get("is_sleep_ctx", pd.Series(False, index=master.index)).fillna(False).astype(bool)

if not reliable:
    hhmm = master.index.strftime("%H:%M")
    wake_mask = (hhmm >= DAY_START) & (hhmm < DAY_END)
    sleep_mask = ~wake_mask

# -------------------------
# Quality summary
# -------------------------
abpm_quality_tbl = pd.DataFrame([{
    "patient_id": patient_id,
    "seconds_total": int(len(master)),
    "pct_analyzable_valid_Intermediate-Quality": _pct(analyzable),
    "pct_valid_only": _pct(valid_only),
    "pct_Intermediate-Quality": _pct(qc.eq("Intermediate-Quality")),
    "pct_artefact": _pct(qc.eq("artefact")),
    "sleepwake_context_reliable": bool(reliable),
    "wake_seconds": _secs(wake_mask),
    "sleep_seconds": _secs(sleep_mask),
    "ABPM_USE_TRIGGER_GUARD": bool(ABPM_USE_TRIGGER_GUARD),
}])

# -------------------------
# Period means
# -------------------------
def _period_stats(name, mask):
    mask = pd.Series(mask, index=master.index).fillna(False).astype(bool)
    base = mask & abpm_base_mask & sbp.notna() & dbp.notna()
    return {
        f"{name}_SBP_mean": _safe_mean(sbp[base]),
        f"{name}_DBP_mean": _safe_mean(dbp[base]),
        f"{name}_MAP_mean": _safe_mean(mapv[base]) if mapv.notna().any() else np.nan,
        f"{name}_HR_mean": _safe_mean(hr[base]) if hr.notna().any() else np.nan,
        f"{name}_SBP_median": _safe_median(sbp[base]),
        f"{name}_DBP_median": _safe_median(dbp[base]),
        f"{name}_n_seconds_used": int(base.sum()),
    }

means = {}
means.update(_period_stats("all24h", pd.Series(True, index=master.index)))
means.update(_period_stats("wake", wake_mask))
means.update(_period_stats("sleep", sleep_mask))

dipping_pct = np.nan
if pd.notna(means.get("wake_SBP_mean", np.nan)) and pd.notna(means.get("sleep_SBP_mean", np.nan)) and means["wake_SBP_mean"] != 0:
    dipping_pct = (means["wake_SBP_mean"] - means["sleep_SBP_mean"]) / means["wake_SBP_mean"] * 100.0

abpm_means_tbl = pd.DataFrame([{
    "patient_id": patient_id,
    **means,
    "SBP_dipping_pct_wake_to_sleep": dipping_pct,
}])

# -------------------------
# Variability (exploratory)
# -------------------------
def _period_var(name, mask):
    mask = pd.Series(mask, index=master.index).fillna(False).astype(bool)
    base = mask & abpm_base_mask & sbp.notna() & dbp.notna()
    return {
        f"{name}_SBP_sd": _safe_sd(sbp[base]),
        f"{name}_DBP_sd": _safe_sd(dbp[base]),
        f"{name}_MAP_sd": _safe_sd(mapv[base]) if mapv.notna().any() else np.nan,
    }

vars_ = {}
vars_.update(_period_var("all24h", pd.Series(True, index=master.index)))
vars_.update(_period_var("wake", wake_mask))
vars_.update(_period_var("sleep", sleep_mask))

abpm_variability_tbl = pd.DataFrame([{"patient_id": patient_id, **vars_}])

# -------------------------
# Blood-pressure load
# Time above or below descriptive thresholds
# -------------------------
HighBP_range = master.get(
    "HighBP_range_sec",
    abpm_base_mask & ((sbp >= CFG["HTN_SBP"]) | (dbp >= CFG["HTN_DBP"]))
).fillna(False).astype(bool)

LowBP_range = master.get(
    "LowBP_range_sec",
    abpm_base_mask & ((sbp <= CFG["HYPOTENSION_SBP"]) | (dbp <= CFG["HYPOTENSION_DBP"]))
).fillna(False).astype(bool)

abpm_load_tbl = pd.DataFrame([{
    "patient_id": patient_id,
    "HighBP_range_minutes": _mins(HighBP_range),
    "LowBP_range_minutes": _mins(LowBP_range),
    "HighBP_range_pct_record": _pct(HighBP_range),
    "LowBP_range_pct_record": _pct(LowBP_range),
    "HighBP_range_pct_valid_only": 100.0 * (HighBP_range & valid_only).sum() / max(1, int(valid_only.sum())),
    "LowBP_range_pct_valid_only": 100.0 * (LowBP_range & valid_only).sum() / max(1, int(valid_only.sum())),
}])

# -------------------------
# Time in state
# Uses pipeline-derived masks; descriptive only
# -------------------------
HighBP_state = master.get("HighBP_state_core", pd.Series(False, index=master.index)).fillna(False).astype(bool)
LowBP_state  = master.get("LowBP_state_core", pd.Series(False, index=master.index)).fillna(False).astype(bool)
BP_drop_local_nonanchored = master.get("BP_drop_local_nonanchored_core", pd.Series(False, index=master.index)).fillna(False).astype(bool)

abpm_state_tbl = pd.DataFrame([{
    "patient_id": patient_id,
    "HighBP_state_minutes": _mins(HighBP_state),
    "LowBP_state_minutes": _mins(LowBP_state),
    "HighBP_state_pct_record": _pct(HighBP_state),
    "LowBP_state_pct_record": _pct(LowBP_state),
    "BP_drop_local_nonanchored_minutes": _mins(BP_drop_local_nonanchored),
}])

# -------------------------
# Combined single-row summary
# -------------------------
abpm_diagnostic_summary_tbl = (
    abpm_quality_tbl
    .merge(abpm_means_tbl, on="patient_id")
    .merge(abpm_variability_tbl, on="patient_id")
    .merge(abpm_load_tbl, on="patient_id")
    .merge(abpm_state_tbl, on="patient_id")
)

print("SECTION 9.2.1 loaded (ABPM-style diagnostic summary tables created).")
print("abpm_quality_tbl"); display(abpm_quality_tbl)
print("abpm_means_tbl"); display(abpm_means_tbl)
print("abpm_variability_tbl"); display(abpm_variability_tbl)
print("abpm_load_tbl"); display(abpm_load_tbl)
print("abpm_state_tbl"); display(abpm_state_tbl)
print("abpm_diagnostic_summary_tbl"); display(abpm_diagnostic_summary_tbl)

# =========================
# Add interpretation string
# Descriptive only; not diagnostic
# =========================

def _fmt(x, nd=1):
    try:
        if pd.isna(x):
            return "NA"
        return f"{float(x):.{nd}f}"
    except Exception:
        return "NA"

def _dipping_label(dip_pct):
    """
    Descriptive dipping categories:
    - reverse_dipping: <0%
    - non_dipping: 0 to <10%
    - dipping: 10 to 20%
    - extreme_dipping: >20%
    """
    if pd.isna(dip_pct):
        return "dipping_na"
    if dip_pct < 0:
        return "reverse_dipping"
    if dip_pct < 10:
        return "non_dipping"
    if dip_pct <= 20:
        return "dipping"
    return "extreme_dipping"

def build_abpm_interpretation(row: pd.Series) -> tuple[str, str]:
    reliable = bool(row.get("sleepwake_context_reliable", False))
    use_guard = bool(row.get("ABPM_USE_TRIGGER_GUARD", True))

    wake_sbp = row.get("wake_SBP_mean", np.nan)
    wake_dbp = row.get("wake_DBP_mean", np.nan)
    sleep_sbp = row.get("sleep_SBP_mean", np.nan)
    sleep_dbp = row.get("sleep_DBP_mean", np.nan)
    all_sbp = row.get("all24h_SBP_mean", np.nan)
    all_dbp = row.get("all24h_DBP_mean", np.nan)

    dip_pct = row.get("SBP_dipping_pct_wake_to_sleep", np.nan)
    dip_lbl = _dipping_label(dip_pct)

    all_sbp_sd = row.get("all24h_SBP_sd", np.nan)
    all_dbp_sd = row.get("all24h_DBP_sd", np.nan)

    high_range_min = row.get("HighBP_range_minutes", np.nan)
    low_range_min = row.get("LowBP_range_minutes", np.nan)
    high_state_min = row.get("HighBP_state_minutes", np.nan)
    low_state_min = row.get("LowBP_state_minutes", np.nan)
    drop_nonanch_min = row.get("BP_drop_local_nonanchored_minutes", np.nan)

    parts = []
    parts.append("ABPM-style summary (descriptive; not diagnostic).")
    parts.append(
        f"Data quality: valid={_fmt(row.get('pct_valid_only', np.nan),1)}%, "
        f"Intermediate-Quality={_fmt(row.get('pct_Intermediate-Quality', np.nan),1)}%, "
        f"artefact={_fmt(row.get('pct_artefact', np.nan),1)}%."
    )
    parts.append(f"Means (24h): {_fmt(all_sbp,1)}/{_fmt(all_dbp,1)} mmHg.")

    if reliable:
        parts.append(
            f"Means (wake): {_fmt(wake_sbp,1)}/{_fmt(wake_dbp,1)}; "
            f"(sleep): {_fmt(sleep_sbp,1)}/{_fmt(sleep_dbp,1)} mmHg; "
            f"SBP dip={_fmt(dip_pct,1)}% ({dip_lbl})."
        )
    else:
        parts.append(
            f"Sleep/Wake context unreliable, so fixed-time fallback was used; "
            f"SBP dip={_fmt(dip_pct,1)}% ({dip_lbl})."
        )

    parts.append(f"Variability (24h SD): SBP={_fmt(all_sbp_sd,1)} mmHg, DBP={_fmt(all_dbp_sd,1)} mmHg.")
    parts.append(f"Load: HighBP_range={_fmt(high_range_min,1)} min; LowBP_range={_fmt(low_range_min,1)} min.")
    parts.append(f"State (>=60s): HighBP_state={_fmt(high_state_min,1)} min; LowBP_state={_fmt(low_state_min,1)} min.")
    parts.append(f"Non-anchored local drop time={_fmt(drop_nonanch_min,1)} min.")
    parts.append("Outlier handling: " + ("L4 trigger-guard applied." if use_guard else "No trigger-guard; QC analyzable mask used."))

    interpretation = " ".join(parts)

    flags = []
    flags.append("ctx_reliable" if reliable else "ctx_unreliable")
    flags.append(dip_lbl)
    if pd.notna(high_state_min) and high_state_min > 30:
        flags.append("highbp_state_gt30min")
    if pd.notna(low_state_min) and low_state_min > 30:
        flags.append("lowbp_state_gt30min")
    if pd.notna(high_range_min) and high_range_min > 60:
        flags.append("highbp_load_gt60min")
    if pd.notna(low_range_min) and low_range_min > 30:
        flags.append("lowbp_load_gt30min")
    flags.append("L4_guard_on" if use_guard else "L4_guard_off")

    return interpretation, ";".join(flags)

interp_list, flags_list = [], []
for _, r in abpm_diagnostic_summary_tbl.iterrows():
    interp, fl = build_abpm_interpretation(r)
    interp_list.append(interp)
    flags_list.append(fl)

abpm_diagnostic_summary_tbl["abpm_interpretation"] = interp_list
abpm_diagnostic_summary_tbl["abpm_flags"] = flags_list

print("Added interpretation string and flags to abpm_diagnostic_summary_tbl.")
pd.set_option("display.max_colwidth", 4000)
display(abpm_diagnostic_summary_tbl[["patient_id", "abpm_interpretation", "abpm_flags"]])

In [ ]:
# =========================
# SECTION 9.2.2 — POSTURE ANCHORING FEASIBILITY (RECORD-LEVEL)
# Quantify whether the Position stream can support posture-anchored OH logic at the record level.
# This section is descriptive not diagnostic.
# =========================

import numpy as np
import pandas as pd

# Prefer pos_meta from Section 3.2 if available; otherwise compute
# a minimal fallback summary from master.
if "pos_meta" in globals() and isinstance(pos_meta, pd.DataFrame) and len(pos_meta):
    posture_feasibility_tbl = pos_meta.copy()
    if "patient_id" not in posture_feasibility_tbl.columns:
        posture_feasibility_tbl.insert(0, "patient_id", patient_id)
else:
    trans = master.get("pos_transition_supine_to_upright", pd.Series(False, index=master.index)).fillna(False).astype(bool)
    anchor = master.get("pos_anchor_strength", pd.Series(0.0, index=master.index))
    postwin = master.get("posture_window_after_upright", pd.Series(False, index=master.index)).fillna(False).astype(bool)

    n_trans = int(trans.sum())
    n_cred = int((trans & (pd.to_numeric(anchor, errors="coerce") >= float(CFG.get("ANCHOR_THR", 0.60)))).sum()) if "pos_anchor_strength" in master.columns else np.nan
    postwin_sec = int(postwin.sum())

    posture_feasibility_tbl = pd.DataFrame([{
        "patient_id": patient_id,
        "pos_known_pct": float(master.get("Position_ctx", pd.Series(index=master.index)).notna().mean() * 100) if "Position_ctx" in master.columns else np.nan,
        "n_transitions_debounced": n_trans,
        "n_credible_transitions": n_cred,
        "posture_window_seconds": postwin_sec,
        "ANCHOR_THR": float(CFG.get("ANCHOR_THR", 0.60)),
        "POS_WIN": np.nan,
        "MIN_STABLE_PCT": np.nan,
        "STATE_GAP_FILL_SEC": np.nan,
        "REFRACTORY_SEC": np.nan,
        "max_supine_prop": np.nan,
        "max_upright_prop": np.nan,
        "pct_windows_sup_ge_07": np.nan,
        "pct_windows_upr_ge_07": np.nan,
    }])

# -------------------------
# Feasibility flag
# Define feasibility as at least one credible transition and at least
# one full OH window of posture coverage.
# -------------------------
posture_window_seconds = float(posture_feasibility_tbl.get("posture_window_seconds", pd.Series([np.nan])).iloc[0])
n_credible = float(posture_feasibility_tbl.get("n_credible_transitions", pd.Series([np.nan])).iloc[0])

OH_WINDOW_SEC = int(CFG.get("OH_WINDOW_SEC", 180))

pos_anchoring_feasible = (
    pd.notna(n_credible) and (n_credible >= 1) and
    pd.notna(posture_window_seconds) and (posture_window_seconds >= OH_WINDOW_SEC)
)

posture_feasibility_tbl["OH_WINDOW_SEC"] = OH_WINDOW_SEC
posture_feasibility_tbl["pos_anchoring_feasible"] = bool(pos_anchoring_feasible)

# -------------------------
# Propagation into run metadata
# -------------------------
if "run_meta_tbl" in globals() and isinstance(run_meta_tbl, pd.DataFrame) and len(run_meta_tbl):
    run_meta_tbl.loc[run_meta_tbl.index[0], "n_transitions_debounced"] = float(posture_feasibility_tbl.get("n_transitions_debounced", pd.Series([np.nan])).iloc[0])
    run_meta_tbl.loc[run_meta_tbl.index[0], "n_credible_transitions"] = float(posture_feasibility_tbl.get("n_credible_transitions", pd.Series([np.nan])).iloc[0])
    run_meta_tbl.loc[run_meta_tbl.index[0], "posture_window_seconds"] = float(posture_feasibility_tbl.get("posture_window_seconds", pd.Series([np.nan])).iloc[0])
    run_meta_tbl.loc[run_meta_tbl.index[0], "pos_anchoring_feasible"] = bool(pos_anchoring_feasible)

print("Posture anchoring feasibility computed.")
display(posture_feasibility_tbl)

In [ ]:
# =========================
# SECTION 9.3 — REVIEWER PLOTS
# QC: Intermediate-Quality=gold, artefact=red
# Effort: purple
# Sleep: light grey
# AD_resting primary: blue
# EFFORT_PRESSOR primary: magenta
# Overlays:
#   HighBP_state (orange), LowBP_state (cyan),
#   BP_drop_local_nonanchored (green),
#   posture_window_after_upright (teal),
#   L4_trigger_guard (black, faint)
# =========================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import numpy as np

def _contiguous_runs(mask: pd.Series) -> pd.DataFrame:
    """
    Convert a boolean mask into contiguous start/end intervals.
    """
    s = pd.Series(mask, index=mask.index).fillna(False).astype(bool)
    if not s.any():
        return pd.DataFrame(columns=["start", "end"])
    grp = s.ne(s.shift()).cumsum()
    rows = []
    for _, block in s.groupby(grp):
        if bool(block.iloc[0]):
            rows.append({"start": block.index[0], "end": block.index[-1]})
    return pd.DataFrame(rows)

def _shade(ax, mask: pd.Series, color: str, alpha: float):
    """
    Shade contiguous True intervals on an axis.
    """
    runs = _contiguous_runs(mask)
    for _, r in runs.iterrows():
        ax.axvspan(r["start"], r["end"], alpha=alpha, color=color, linewidth=0)

def plot_reviewer_timeline(
    master: pd.DataFrame,
    final_sets: dict,
    *,
    patient_id: str = "",
    title_extra: str = "",
    resample_sec: int = 5,
    show_sleep: bool = True,
    show_bp_states: bool = True,
    show_drop_nonanchored: bool = True,
    show_posture_window: bool = True,
    show_L4_exclusions: bool = True
):
    """
    Plot a compact multi-panel reviewer timeline with signal traces,
    QC shading, context overlays, and primary event overlays.
    """
    rs = f"{resample_sec}s"

    # Resampled numeric signals
    cols = [c for c in ["SBP", "HR", "PTTraw", "Activity"] if c in master.columns]
    plot_num = master[cols].resample(rs).median()

    # QC mode per resampled bin
    if "QC_status" in master.columns:
        plot_qc = master["QC_status"].resample(rs).agg(
            lambda x: pd.Series(x).dropna().mode().iloc[0] if len(pd.Series(x).dropna().mode()) else "valid"
        )
    else:
        plot_qc = pd.Series("valid", index=plot_num.index)

    # Resampled masks
    eff = master.get("effort_context_mask", pd.Series(False, index=master.index)).resample(rs).max().fillna(False).astype(bool)

    if show_sleep:
        sleep_mask = master.get("is_sleep_ctx", pd.Series(False, index=master.index)).resample(rs).max().fillna(False).astype(bool)
    else:
        sleep_mask = pd.Series(False, index=plot_num.index)

    if show_bp_states:
        high_state = master.get("HighBP_state_core", pd.Series(False, index=master.index)).resample(rs).max().fillna(False).astype(bool)
        low_state  = master.get("LowBP_state_core",  pd.Series(False, index=master.index)).resample(rs).max().fillna(False).astype(bool)
    else:
        high_state = pd.Series(False, index=plot_num.index)
        low_state  = pd.Series(False, index=plot_num.index)

    if show_drop_nonanchored:
        drop_nonanch = master.get("BP_drop_local_nonanchored_core", pd.Series(False, index=master.index)).resample(rs).max().fillna(False).astype(bool)
    else:
        drop_nonanch = pd.Series(False, index=plot_num.index)

    if show_posture_window:
        post_win = master.get("posture_window_after_upright", pd.Series(False, index=master.index)).resample(rs).max().fillna(False).astype(bool)
    else:
        post_win = pd.Series(False, index=plot_num.index)

    if show_L4_exclusions:
        l4 = master.get("L4_trigger_guard", pd.Series(False, index=master.index)).resample(rs).max().fillna(False).astype(bool)
    else:
        l4 = pd.Series(False, index=plot_num.index)

    # Figure layout
    fig, axes = plt.subplots(4, 1, figsize=(18, 10), sharex=True, constrained_layout=True)

    for ax in axes:
        _shade(ax, plot_qc.eq("Intermediate-Quality"), "gold", 0.10)
        _shade(ax, plot_qc.eq("artefact"), "red", 0.10)
        _shade(ax, eff, "purple", 0.06)
        _shade(ax, sleep_mask, "lightgrey", 0.06)

        if show_bp_states:
            _shade(ax, high_state, "orange", 0.04)
            _shade(ax, low_state, "cyan", 0.04)

        if show_drop_nonanchored:
            _shade(ax, drop_nonanch, "green", 0.03)

        if show_posture_window:
            _shade(ax, post_win, "teal", 0.03)

        if show_L4_exclusions:
            _shade(ax, l4, "black", 0.02)

    # Signal traces
    if "SBP" in plot_num.columns:
        axes[0].plot(plot_num.index, plot_num["SBP"], lw=1)
    axes[0].set_ylabel("SBP")

    if "HR" in plot_num.columns:
        axes[1].plot(plot_num.index, plot_num["HR"], lw=1)
    axes[1].set_ylabel("HR")

    if "PTTraw" in plot_num.columns:
        axes[2].plot(plot_num.index, plot_num["PTTraw"], lw=1)
    axes[2].set_ylabel("PTTraw")

    if "Activity" in plot_num.columns:
        axes[3].plot(plot_num.index, plot_num["Activity"], lw=1)
    axes[3].set_ylabel("Activity")

    # Primary event overlays
    ad_prim = final_sets.get("AD_resting", {}).get("primary", pd.DataFrame())
    eff_prim = final_sets.get("EFFORT_PRESSOR", {}).get("primary", pd.DataFrame())

    for _, ev in ad_prim.iterrows():
        axes[0].axvspan(pd.to_datetime(ev["start"]), pd.to_datetime(ev["end"]), alpha=0.25, color="dodgerblue")
    for _, ev in eff_prim.iterrows():
        axes[0].axvspan(pd.to_datetime(ev["start"]), pd.to_datetime(ev["end"]), alpha=0.20, color="magenta")

    # X-axis formatting
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    # Title
    legend_hint = "QC(gold/red) Effort(purple) Sleep(grey) AD_resting(blue) EffortPressor(magenta)"
    extras = []
    if show_bp_states:
        extras.append("BP_state(orange/cyan)")
    if show_drop_nonanchored:
        extras.append("Drop_nonanch(green)")
    if show_posture_window:
        extras.append("PostWin(teal)")
    if show_L4_exclusions:
        extras.append("L4_excl(black)")
    extra_txt = (" | " + ", ".join(extras)) if extras else ""

    t = f"{patient_id} — {legend_hint}{extra_txt}"
    if title_extra:
        t += f" — {title_extra}"
    fig.suptitle(t, y=1.02)

    plt.show()

# Default call for the current headline run
plot_reviewer_timeline(master, final_sets, patient_id=patient_id, resample_sec=5, title_extra="headline baseline")
print("SECTION 9.3 plotted.")

In [ ]:
# =========================
# SECTION 10 — EXPORT, ZIP, AND DOWNLOAD
# Rebuild export_pack from the current run tables.
# Core tables must match the canonical patient identifier.
# =========================

import os, json, shutil
from datetime import datetime, timezone
import pandas as pd
import numpy as np

OUT_ROOT = "/content/outputs"
OUT_DIR = os.path.join(OUT_ROOT, str(patient_id))

# -------------------------
# Reset output folder to avoid mixing files from previous runs
# -------------------------
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

QUARANTINE_DIR = os.path.join(OUT_DIR, "_STALE_QUARANTINE")
os.makedirs(QUARANTINE_DIR, exist_ok=True)

# -------------------------
# Helper functions
# -------------------------
def _pid_set(df):
    """
    Return the set of unique patient identifiers present in a dataframe.
    """
    if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
        return set()
    if "patient_id" not in df.columns:
        return set()
    return set(df["patient_id"].dropna().astype(str).unique())

def _canonical_pid():
    """
    Resolve the canonical patient identifier for export.

    Preference order:
    1. event_tbl
    2. qc_tbl
    3. abpm_diagnostic_summary_tbl
    4. global patient_id
    """
    for name in ["event_tbl", "qc_tbl", "abpm_diagnostic_summary_tbl"]:
        if name in globals():
            s = _pid_set(globals()[name])
            if len(s) == 1:
                return list(s)[0]
    return str(patient_id)

PID = _canonical_pid()

# Report any mismatch between the global variable and canonical export PID.
if str(patient_id) != str(PID):
    print(
        f"Note: folder patient_id variable='{patient_id}' but canonical PID from tables='{PID}'. "
        f"Export tables will be validated against PID='{PID}'."
    )

def save_df_csv(df, filename, *, expected_pid=None, name="table",
                strict=True, empty_cols=None, quarantine_on_fail=True):
    """
    Save a dataframe to CSV with optional patient_id validation.

    strict=True  -> mismatched patient_id raises an error
    strict=False -> mismatched patient_id is skipped
    """
    path = os.path.join(OUT_DIR, filename)

    # Write an empty file if the table is missing or empty.
    if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
        if empty_cols is not None:
            pd.DataFrame(columns=empty_cols).to_csv(path, index=False)
        else:
            pd.DataFrame().to_csv(path, index=False)
        return "empty"

    # Validate patient_id when requested.
    if expected_pid is not None and "patient_id" in df.columns:
        vals = _pid_set(df)
        if len(vals) > 0 and vals != {str(expected_pid)}:
            msg = f"[EXPORT {'ERROR' if strict else 'SKIP'}] {name} has patient_id={vals} but expected={expected_pid}"
            if quarantine_on_fail:
                qpath = os.path.join(QUARANTINE_DIR, f"{filename.replace('.csv','')}_STALE_{list(vals)[0]}.csv")
                df.to_csv(qpath, index=False)
                msg += f" | quarantined -> {qpath}"
            if strict:
                raise RuntimeError(msg)
            else:
                print("⚠️", msg)
                return "skipped"

    df.to_csv(path, index=False)
    return "saved"

def stack_dict_of_dfs(d: dict, pid: str, label_col: str = "event_type") -> pd.DataFrame:
    """
    Stack a dictionary of dataframes into a single long-form dataframe,
    stamping each component with patient_id and event_type.
    """
    rows = []
    if not isinstance(d, dict):
        return pd.DataFrame(columns=["patient_id", label_col])
    for name, df in d.items():
        if df is None or not isinstance(df, pd.DataFrame) or len(df) == 0:
            continue
        tmp = df.copy()
        if "patient_id" not in tmp.columns:
            tmp.insert(0, "patient_id", pid)
        tmp.insert(1, label_col, name)
        rows.append(tmp)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["patient_id", label_col])

# -------------------------
# Rebuild export_pack from current run objects
# -------------------------
export_pack = {
    "baseline_model_tbl": baseline_model_tbl if "baseline_model_tbl" in globals() else None,
    "baseline_candidates": baseline_candidates if "baseline_candidates" in globals() else None,
    "baseline_sensitivity_tbl": baseline_sensitivity_tbl if "baseline_sensitivity_tbl" in globals() else None,
    "baseline_final_summary_tbl": baseline_final_summary_tbl if "baseline_final_summary_tbl" in globals() else None,
    "diurnal_tbl": diurnal_tbl if "diurnal_tbl" in globals() else None,
    "posture_feasibility_tbl": posture_feasibility_tbl if "posture_feasibility_tbl" in globals() else None,
    "abpm_diagnostic_summary_tbl": abpm_diagnostic_summary_tbl if "abpm_diagnostic_summary_tbl" in globals() else None,
}

if "protocol_tbl" in globals():
    export_pack["protocol_tbl"] = protocol_tbl
if "protocol_summary" in globals():
    export_pack["protocol_summary"] = protocol_summary

# -------------------------
# Core tables: enforce strict PID validation
# -------------------------
save_df_csv(qc_tbl, "qc_tbl.csv", expected_pid=PID, name="qc_tbl", strict=True)
save_df_csv(event_tbl, "event_tbl.csv", expected_pid=PID, name="event_tbl", strict=True)
save_df_csv(run_meta_tbl, "run_meta_tbl.csv", expected_pid=PID, name="run_meta_tbl", strict=True)

# -------------------------
# Tables: skip if stale
# -------------------------
save_df_csv(export_pack.get("baseline_model_tbl"), "baseline_model_tbl.csv", expected_pid=PID, name="baseline_model_tbl", strict=False)
save_df_csv(export_pack.get("baseline_sensitivity_tbl"), "baseline_sensitivity_tbl.csv", expected_pid=PID, name="baseline_sensitivity_tbl", strict=False)
save_df_csv(export_pack.get("baseline_final_summary_tbl"), "baseline_final_summary_tbl.csv", expected_pid=PID, name="baseline_final_summary_tbl", strict=False)
save_df_csv(export_pack.get("diurnal_tbl"), "diurnal_tbl.csv", expected_pid=PID, name="diurnal_tbl", strict=False)
save_df_csv(export_pack.get("posture_feasibility_tbl"), "posture_feasibility_tbl.csv", expected_pid=PID, name="posture_feasibility_tbl", strict=False)
save_df_csv(export_pack.get("abpm_diagnostic_summary_tbl"), "abpm_diagnostic_summary_tbl.csv", expected_pid=PID, name="abpm_diagnostic_summary_tbl", strict=False)

# baseline_candidates may not contain patient_id, so export without validation
save_df_csv(export_pack.get("baseline_candidates"), "baseline_candidates.csv", name="baseline_candidates", strict=False)

# Protocol tables are exported without patient_id validation
if export_pack.get("protocol_tbl") is not None:
    save_df_csv(export_pack["protocol_tbl"], "protocol_tbl.csv", name="protocol_tbl", strict=False)
if export_pack.get("protocol_summary") is not None:
    save_df_csv(export_pack["protocol_summary"], "protocol_summary.csv", name="protocol_summary", strict=False)

# -------------------------
# Long tables generated from current run objects
# -------------------------
episodes_all = stack_dict_of_dfs(episode_tables, PID, "event_type") if "episode_tables" in globals() else pd.DataFrame(columns=["patient_id", "event_type"])
adjudication_all = stack_dict_of_dfs(review_tables, PID, "event_type") if "review_tables" in globals() else pd.DataFrame(columns=["patient_id", "event_type"])
final_primary_all = (
    stack_dict_of_dfs({k: v.get("primary", pd.DataFrame()) for k, v in final_sets.items()}, PID, "event_type")
    if "final_sets" in globals() else pd.DataFrame(columns=["patient_id", "event_type"])
)
final_sustained_all = (
    stack_dict_of_dfs({k: v.get("sustained", pd.DataFrame()) for k, v in final_sets.items()}, PID, "event_type")
    if "final_sets" in globals() else pd.DataFrame(columns=["patient_id", "event_type"])
)

save_df_csv(episodes_all, "episodes_all.csv", expected_pid=PID, name="episodes_all", strict=True, empty_cols=["patient_id", "event_type"])
save_df_csv(adjudication_all, "adjudication_all.csv", expected_pid=PID, name="adjudication_all", strict=True, empty_cols=["patient_id", "event_type"])
save_df_csv(final_primary_all, "final_primary.csv", expected_pid=PID, name="final_primary", strict=True, empty_cols=["patient_id", "event_type"])
save_df_csv(final_sustained_all, "final_sustained.csv", expected_pid=PID, name="final_sustained", strict=True, empty_cols=["patient_id", "event_type"])

# -------------------------
# Manifest
# -------------------------
manifest = {
    "patient_id": str(PID),
    "patient_id_variable_at_export": str(patient_id),
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "CFG_VERSION": CFG_VERSION if "CFG_VERSION" in globals() else None,
    "seconds_total": int(len(master)) if "master" in globals() else None,
    "baseline_ref_source": (
        str(master["baseline_ref_source"].dropna().iloc[0])
        if "master" in globals() and "baseline_ref_source" in master.columns and master["baseline_ref_source"].notna().any()
        else None
    ),
    "baseline_ref_SBP": (
        float(master["baseline_ref_SBP"].dropna().iloc[0])
        if "master" in globals() and "baseline_ref_SBP" in master.columns and master["baseline_ref_SBP"].notna().any()
        else None
    ),
    "L4_trigger_guard_seconds": (
        int(master.get("L4_trigger_guard", pd.Series(False, index=master.index)).sum())
        if "master" in globals() else None
    ),
    "stale_quarantine_dir": QUARANTINE_DIR,
}
with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

# -------------------------
# Zip file and download
# -------------------------
zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print("Created ZIP:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download not triggered (not in Colab?):", e)

print("Export complete for canonical PID:", PID)
print("Files saved in:", OUT_DIR)
print("Quarantine (if any):", QUARANTINE_DIR)